# 관측 두께·정답 스켈레톤으로 목표 이미지를 만들고 재역확산

GPU 런타임(T4 가능)에서 **코드 셀 1→5** 실행. 학습·새 2×2 생성 없이 기존 생성물을 사용합니다.
2번 셀에서 **repair_stage4_20261123_132010_265667.zip**을 업로드하세요.
기존 Drive의 best.pt도 필요합니다. 기본은 첫 생성 시트와 대응 정상 시트 **2장**만 비교합니다.

획 추출 → 관측 두께 → 정답 스켈레톤 렌더링 → 글자 공통 affine → 획별 수정 강도 → 원본에서 재역확산.
비교는 입력 / 목표(확산 아님) / 일반 재역확산 / 목표 가이드 재역확산입니다.
두 확산 경로는 같은 초기 latent를 사용합니다. 최종 목표 이미지 덮어쓰기는 없습니다.

**점수 주의:** 외부 점수가 없으면 미검증 기하 지표를 사용합니다. 논문 평가기의 재현이나 정오 판정기가 아닙니다.
앞선 CPU 사례에서 24획 중 23획을 수정 대상으로 잡았으므로 정상 대조군의 과잉 수정을 함께 확인하세요.
60 nominal steps × strength .25 = 경로당 실제 15스텝, 시트당 두 경로입니다. GPU 품질은 이번에 검증합니다.


In [ ]:
# 1. 설치 및 고정 런타임 준비. 이 노트북만으로 실행 가능
import sys, subprocess, json, hashlib, gc, shutil, traceback, time, zipfile
from pathlib import Path
from datetime import datetime
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'diffusers==0.37.0', 'transformers==4.57.6', 'accelerate', 'safetensors',
    'timm', 'numpy', 'scipy', 'scikit-image', 'pillow'], check=True)
RUNTIME_HELPERS={'stroke_model/reference_model.py': '"""Reference-conditioned V2. Inputs: white ink on black, Bx1xHxW in [0,1].\n\nNo thresholding, CPU conversion or detached image features in the forward path.\nOld checkpoints are deliberately not compatible with this new architecture.\n"""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef block(cin, cout, stride=1):\n    return nn.Sequential(nn.Conv2d(cin, cout, 3, stride=stride, padding=1),\n                         nn.GroupNorm(4, cout), nn.GELU())\n\n\ndef resize(x, size):\n    return F.interpolate(x, size=size, mode="bilinear", align_corners=False)\n\n\ndef warp(strokes, flow):\n    b, _, h, w = strokes.shape\n    ys = (torch.arange(h, device=strokes.device, dtype=strokes.dtype) + .5) * 2 / h - 1\n    xs = (torch.arange(w, device=strokes.device, dtype=strokes.dtype) + .5) * 2 / w - 1\n    y, x = torch.meshgrid(ys, xs, indexing="ij")\n    grid = torch.stack((x, y), -1)[None].expand(b, -1, -1, -1)\n    offset = torch.stack((flow[:, 0] * 2 / w, flow[:, 1] * 2 / h), -1)\n    return F.grid_sample(strokes, grid + offset, align_corners=False)\n\n\nclass ReferenceStrokeModel(nn.Module):\n    def __init__(self, image_size=128, width=64, pretrained=True,\n                 backbone=None, channels=None, channels_last=False, stroke_chunk_size=4):\n        super().__init__()\n        if width % 4:\n            raise ValueError("width must be divisible by 4")\n        mean, std = (.485, .456, .406), (.229, .224, .225)\n        if backbone is None:\n            import timm\n            backbone = timm.create_model(\n                "swin_tiny_patch4_window7_224", pretrained=pretrained,\n                features_only=True, out_indices=(0, 1, 2, 3),\n                in_chans=3, img_size=image_size, drop_path_rate=0.)\n            channels = backbone.feature_info.channels()\n            channels_last = True  # timm Swin features are NHWC.\n            mean = backbone.pretrained_cfg.get("mean", mean)\n            std = backbone.pretrained_cfg.get("std", std)\n        if not channels:\n            raise ValueError("Injected backbone requires its feature channels")\n        self.backbone = backbone\n        self.stroke_chunk_size = max(1, int(stroke_chunk_size))\n        self.channels_last = channels_last\n        self.backbone_frozen = True\n        self.backbone.requires_grad_(False)\n        self.backbone.eval()\n        self.register_buffer("rgb_mean", torch.tensor(mean).view(1, 3, 1, 1))\n        self.register_buffer("rgb_std", torch.tensor(std).view(1, 3, 1, 1))\n        self.lateral = nn.ModuleList(nn.Conv2d(c, width, 1) for c in channels)\n        self.smooth = nn.ModuleList(block(width, width) for _ in channels)\n        self.detail = block(1, width)  # Full-resolution branch for thin strokes.\n        self.fuse = block(width * 2, width)\n        self.flow_net = nn.Sequential(block(2, 16, 2), block(16, 32, 2),\n                                      nn.Conv2d(32, 2, 3, padding=1))\n        nn.init.zeros_(self.flow_net[-1].weight)\n        nn.init.zeros_(self.flow_net[-1].bias)\n        # Shared across stroke IDs; full glyph + selected stroke + xy context.\n        self.reference = nn.Sequential(block(4, 16, 2), block(16, width, 2))\n        self.head = nn.Sequential(block(width * 2 + 4, width),\n                                  nn.Conv2d(width, 1, 1))\n\n    def train(self, mode=True):\n        super().train(mode)\n        if self.backbone_frozen:\n            self.backbone.eval()\n        return self\n\n    def forward(self, image, reference, strokes, active=None):\n        if image.ndim != 4 or image.shape[1] != 1 or reference.shape != image.shape:\n            raise ValueError("image/reference must have matching Bx1xHxW shapes")\n        b, _, h, w = image.shape\n        if strokes.ndim != 4 or strokes.shape[0] != b or strokes.shape[-2:] != (h, w):\n            raise ValueError("strokes must have shape BxKxHxW")\n        k = strokes.shape[1]\n        if active is None:\n            active = strokes.flatten(2).sum(-1) > 0  # Fixed reference metadata only.\n        if active.shape != (b, k) or active.dtype != torch.bool or not active.any(1).all():\n            raise ValueError("Each sample needs at least one active reference stroke")\n        rgb = (1 - image).expand(-1, 3, -1, -1)\n        feats = self.backbone((rgb - self.rgb_mean) / self.rgb_std)\n        if self.channels_last:\n            feats = [f.permute(0, 3, 1, 2) for f in feats]\n        pyramid = None\n        for i in reversed(range(len(feats))):\n            lateral = self.lateral[i](feats[i])\n            pyramid = lateral if pyramid is None else lateral + resize(pyramid, lateral.shape[-2:])\n            pyramid = self.smooth[i](pyramid)\n        features = self.fuse(torch.cat((resize(pyramid, (h, w)), self.detail(image)), 1))\n        flow = 12 * resize(self.flow_net(torch.cat((reference, image), 1)), (h, w)).tanh()\n        hints = warp(strokes, flow)\n        yy, xx = torch.meshgrid(torch.linspace(-1, 1, h, device=image.device, dtype=image.dtype),\n                                torch.linspace(-1, 1, w, device=image.device, dtype=image.dtype), indexing="ij")\n        xy = torch.stack((xx, yy))[None].expand(b, -1, -1, -1)\n        logits = self.score_strokes(features, reference, strokes, xy, flow, hints, image)\n        logits = logits.float().masked_fill(~active[:, :, None, None], -torch.inf)\n        probabilities = logits.softmax(1) * active[:, :, None, None]\n        probabilities = probabilities / probabilities.sum(1, keepdim=True)\n        return {"logits": logits, "probabilities": probabilities,\n                "masks": image * probabilities, "flow": flow, "hints": hints,\n                "active": active}\n\n    def score_strokes(self, features, reference, strokes, xy, flow, hints, image):\n        """Same head algebra, batching strokes and computing shared convolution once.\n\n        GroupNorm is per sample, so stroke batching does not mix stroke statistics.\n        Existing V2 state_dicts remain compatible; input gradients remain connected.\n        """\n        b, k, h, w = strokes.shape\n        width = features.shape[1]\n        conv = self.head[0][0]\n        shared_weight = torch.cat((conv.weight[:, :width], conv.weight[:, 2*width+1:]), 1)\n        shared = F.conv2d(torch.cat((features, image, xy), 1), shared_weight, conv.bias, padding=1)\n        outputs = []\n        for start in range(0, k, self.stroke_chunk_size):\n            n = min(self.stroke_chunk_size, k-start)\n            def repeat(t):\n                return t[:, None].expand(-1, n, -1, -1, -1).reshape(b*n, *t.shape[1:])\n            selected = strokes[:, start:start+n].reshape(b*n, 1, h, w)\n            r = self.reference(torch.cat((repeat(reference), selected, repeat(xy)), 1))\n            r = warp(resize(r, (h, w)), repeat(flow))\n            condition = torch.cat((r, hints[:, start:start+n].reshape(b*n, 1, h, w)), 1)\n            x = F.conv2d(condition, conv.weight[:, width:2*width+1], padding=1) + repeat(shared)\n            x = self.head[0][2](self.head[0][1](x))\n            outputs.append(self.head[1](x).reshape(b, n, h, w))\n        return torch.cat(outputs, 1)\n\n\ndef exclusive_masks(output, foreground):\n    """Export only: non-differentiable exact partition of the supplied foreground."""\n    expected = output[\'probabilities\'][:, :1].shape\n    if foreground.shape != expected:\n        raise ValueError(\'foreground must have shape Bx1xHxW\')\n    labels = output["probabilities"].argmax(1)\n    masks = F.one_hot(labels, output["probabilities"].shape[1]).permute(0, 3, 1, 2)\n    return masks.bool() & foreground.bool()\n\n\ndef stroke_moments(masks, eps=1e-6):\n    """Area fraction, centroid x/y and xx/xy/yy covariance in normalized coordinates.\n\n    Near-empty strokes have unreliable geometry: callers must mask/inspect them.\n    """\n    h, w = masks.shape[-2:]\n    y, x = torch.meshgrid(torch.linspace(-1, 1, h, device=masks.device, dtype=masks.dtype),\n                          torch.linspace(-1, 1, w, device=masks.device, dtype=masks.dtype), indexing="ij")\n    mass = masks.sum((-2, -1))\n    weights = masks / mass.clamp_min(eps)[..., None, None]\n    cx, cy = (weights * x).sum((-2, -1)), (weights * y).sum((-2, -1))\n    dx, dy = x - cx[..., None, None], y - cy[..., None, None]\n    return torch.stack((mass / (h * w), cx, cy,\n                        (weights * dx.square()).sum((-2, -1)),\n                        (weights * dx * dy).sum((-2, -1)),\n                        (weights * dy.square()).sum((-2, -1))), -1)\n\n\ndef partition_loss(output, labels):\n    """labels: BxHxW, 0..K-1 for owned pixels; -1 background/unknown.\n\n    Unknown crossing pixels are excluded, not assigned pseudo-certainty.\n    Soft Dice averages only annotated, nonempty target strokes.\n    """\n    logits = output["logits"]\n    valid = labels >= 0\n    if labels.shape != logits.shape[:1] + logits.shape[2:]:\n        raise ValueError("labels must have shape BxHxW")\n    if not valid.any() or (labels < -1).any() or (labels >= logits.shape[1]).any():\n        raise ValueError("Need annotated pixels with labels in -1..K-1")\n    target = F.one_hot(labels.clamp_min(0).long(), logits.shape[1]).permute(0, 3, 1, 2)\n    target = target * valid[:, None]\n    if (target.bool() & ~output["active"][:, :, None, None]).any():\n        raise ValueError("Target refers to an inactive reference stroke")\n    ce = F.cross_entropy(logits, labels.long(), ignore_index=-1)\n    pred = output["probabilities"] * valid[:, None]\n    mass = target.sum((-2, -1))\n    dice = (2 * (pred * target).sum((-2, -1)) + 1e-6) / (pred.sum((-2, -1)) + mass + 1e-6)\n    dice_loss = (1 - dice)[mass > 0].mean()\n    flow = output["flow"]\n    smooth = (flow[..., 1:, :] - flow[..., :-1, :]).square().mean()\n    smooth = smooth + (flow[..., :, 1:] - flow[..., :, :-1]).square().mean()\n    return {"loss": ce + dice_loss + .001 * smooth, "ce": ce,\n            "dice_loss": dice_loss, "flow_smoothness": smooth}\n', 'stroke_model/stroke_structure.py': '"""Reference-aware experimental stroke losses; inputs are absolute ink in [0, 1].\n\nReference preparation is discrete and cached. All prediction-dependent losses use\nPyTorch operations, including min/max subgradients. No predicted components are\ndiscarded. Scores are uncalibrated errors, not human-quality probabilities.\n"""\nfrom dataclasses import dataclass, fields\nimport math\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom scipy.ndimage import distance_transform_edt, label\nfrom skimage.morphology import skeletonize\n\n\n@dataclass(frozen=True)\nclass StructureConfig:\n    # All distances are pixels at the inference resolution (normally 128).\n    reference_threshold: float = 0.5\n    coverage_radius: int = 2\n    required_ink: float = 0.7\n    anchor_spacing: int = 10\n    anchor_radius: int = 1\n    corridor_radius: int = 3\n    pair_spans: tuple = (1, 2)\n    max_pairs: int = 32\n    connectivity_steps: int | None = None  # None: crop area, with convergence check\n    skeleton_iterations: int = 12\n    junction_radii: tuple = (1, 2, 3)\n    junction_margin: float = 0.0  # No font/short-branch exemption by default\n    junction_thresholds: tuple = (0.15, 0.30, 0.45, 0.60, 0.75)\n    junction_temperature: float = 0.025  # Soft occupancy transition, absolute ink\n    junction_arm_scale: float = 1.25\n    junction_arm_offset: float = 1.0\n    junction_min_arm_length: float = 2.0\n    junction_merge_radius: int = 1\n    junction_match_radius: int = 2\n    junction_min_levels: int = 2\n    junction_probe_scale: float = 1.5\n    junction_probe_offset: float = 1.0\n    junction_distance_steps: int = 12\n    junction_angular_samples: int = 48\n    junction_angular_smoothing: int = 3\n    junction_ring_margin: float = 0.1\n    junction_seed_radius: int = 2\n    attachment_steps: int | None = None  # None: H*W, with convergence check\n    expected_width: float = 5.0  # Optional normal-width prior, NOT skeleton width\n    width_radius: int = 4\n    width_tolerance: float = 0.25\n    missing_weight: float = 1.0\n    split_weight: float = 1.0\n    junction_weight: float = 1.0\n    width_weight: float = 0.0  # Enable only after normal-width calibration\n    eps: float = 1e-6\n\n    def __post_init__(self):\n        for name in (\'coverage_radius\', \'anchor_radius\', \'corridor_radius\',\n                     \'junction_seed_radius\', \'width_radius\'):\n            if not isinstance(getattr(self, name), int) or getattr(self, name) < 0:\n                raise ValueError(f\'{name} must be a nonnegative integer\')\n        for name in (\'anchor_spacing\', \'max_pairs\', \'skeleton_iterations\'):\n            if not isinstance(getattr(self, name), int) or getattr(self, name) < 1:\n                raise ValueError(f\'{name} must be a positive integer\')\n        if self.anchor_spacing <= 2 * self.anchor_radius + 1:\n            raise ValueError(\'anchor_spacing must exceed the anchor diameters\')\n        if not self.pair_spans or any(not isinstance(v, int) or v < 1 for v in self.pair_spans):\n            raise ValueError(\'pair_spans must contain positive integers\')\n        if not self.junction_radii or any(not isinstance(v, int) or v < 1 for v in self.junction_radii):\n            raise ValueError(\'junction_radii must contain integers >= 1\')\n        if (not self.junction_thresholds or any(not math.isfinite(float(v)) or not 0 < v < 1\n                                               for v in self.junction_thresholds)\n                or sorted(set(self.junction_thresholds)) != list(self.junction_thresholds)):\n            raise ValueError(\'junction_thresholds must be strictly increasing values in (0,1)\')\n        if not math.isfinite(self.junction_temperature) or self.junction_temperature <= 0:\n            raise ValueError(\'junction_temperature must be finite and positive\')\n        for name in (\'junction_arm_scale\', \'junction_min_arm_length\', \'junction_probe_scale\'):\n            if not math.isfinite(getattr(self, name)) or getattr(self, name) <= 0:\n                raise ValueError(f\'{name} must be positive\')\n        for name in (\'junction_arm_offset\', \'junction_probe_offset\', \'junction_ring_margin\'):\n            if not math.isfinite(getattr(self, name)) or getattr(self, name) < 0:\n                raise ValueError(f\'{name} must be nonnegative\')\n        for name in (\'junction_merge_radius\', \'junction_match_radius\', \'junction_angular_smoothing\'):\n            if not isinstance(getattr(self, name), int) or getattr(self, name) < 0:\n                raise ValueError(f\'{name} must be a nonnegative integer\')\n        if (not isinstance(self.junction_min_levels, int)\n                or not 1 <= self.junction_min_levels <= len(self.junction_thresholds)):\n            raise ValueError(\'junction_min_levels must fit the threshold count\')\n        if not isinstance(self.junction_distance_steps, int) or self.junction_distance_steps < 1:\n            raise ValueError(\'junction_distance_steps must be a positive integer\')\n        if not isinstance(self.junction_angular_samples, int) or self.junction_angular_samples < 16:\n            raise ValueError(\'junction_angular_samples must be an integer >= 16\')\n        for name in (\'connectivity_steps\', \'attachment_steps\'):\n            v = getattr(self, name)\n            if v is not None and (not isinstance(v, int) or v < 1):\n                raise ValueError(f\'{name} must be None or a positive integer\')\n        if not 0 < self.required_ink <= 1 or not 0 < self.reference_threshold < 1:\n            raise ValueError(\'required_ink must be in (0,1]; reference_threshold in (0,1)\')\n        if self.expected_width <= 0 or self.eps <= 0 or not 0 <= self.width_tolerance < 1:\n            raise ValueError(\'Invalid width or eps\')\n        if self.junction_margin < 0 or any(getattr(self, n) < 0 for n in\n                (\'missing_weight\', \'split_weight\', \'junction_weight\', \'width_weight\')):\n            raise ValueError(\'Weights and junction_margin must be nonnegative\')\n        if any(not math.isfinite(float(getattr(self, f.name))) for f in fields(self)\n               if isinstance(getattr(self, f.name), (float, int))):\n            raise ValueError(\'Configuration values must be finite\')\n\n\ndef _pool(x, radius, mean=False):\n    k = 2 * radius + 1\n    return (F.avg_pool2d(x, k, 1, radius) if mean else F.max_pool2d(x, k, 1, radius))\n\n\ndef soft_skeleton(x, iterations):\n    """Pooling skeleton from soft-clDice; a surrogate, not exact graph thinning."""\n    def erode(v):\n        return torch.minimum(-F.max_pool2d(-v, (3, 1), 1, (1, 0)),\n                             -F.max_pool2d(-v, (1, 3), 1, (0, 1)))\n    def opening(v):\n        return F.max_pool2d(erode(v), 3, 1, 1)\n    skel = F.relu(x - opening(x))\n    for _ in range(iterations):\n        x = erode(x)\n        delta = F.relu(x - opening(x))\n        skel = skel + F.relu(delta - skel * delta)\n    return skel\n\n\ndef reconstruct(mask, seed, steps=None):\n    """8-connected grayscale reconstruction; no dilation/closing of mask.\n\n    Returns (reachable strength, converged). None allows H*W iterations, a\n    conservative finite-grid bound. Early stopping reads detached values only;\n    the returned tensor retains the graph. Explicit small step caps can truncate.\n    """\n    reached = torch.minimum(mask, seed)\n    limit = mask.shape[-2] * mask.shape[-1] if steps is None else steps\n    converged = False\n    for _ in range(limit):\n        updated = torch.minimum(mask, F.max_pool2d(reached, 3, 1, 1))\n        if torch.equal(updated.detach(), reached.detach()):\n            reached = updated\n            converged = True\n            break\n        reached = updated\n    return reached, converged\n\n\ndef _ordered_centerline(binary):\n    skel = skeletonize(binary)\n    _, count = label(skel, structure=np.ones((3, 3)))\n    if count != 1:\n        raise ValueError(\'Reference stroke must have one connected centerline; clean the reference\')\n    points = [tuple(v) for v in np.argwhere(skel)]\n    lookup = set(points)\n    neighbors = {}\n    for y, x in points:\n        adjacent = []\n        for dy in (-1, 0, 1):\n            for dx in (-1, 0, 1):\n                q = (y + dy, x + dx)\n                if (not dy and not dx) or q not in lookup:\n                    continue\n                # Avoid triangular shortcut edges at a normal digital corner.\n                if dy and dx and ((y + dy, x) in lookup or (y, x + dx) in lookup):\n                    continue\n                adjacent.append(q)\n        neighbors[(y, x)] = adjacent\n    if any(len(v) > 2 for v in neighbors.values()):\n        raise ValueError(\'Reference skeleton branches; provide a clean unbranched reference stroke\')\n    endpoints = sorted(p for p, n in neighbors.items() if len(n) == 1)\n    closed = len(points) > 2 and not endpoints\n    if len(points) < 5 or (not closed and len(endpoints) != 2):\n        raise ValueError(\'Reference centerline must be a nontrivial open curve or closed ring\')\n    start = endpoints[0] if endpoints else min(points)\n    order, visited, current = [], set(), start\n    while current not in visited:\n        order.append(current)\n        visited.add(current)\n        choices = sorted(q for q in neighbors[current] if q not in visited)\n        if not choices:\n            break\n        current = choices[0]\n    if len(order) != len(points):\n        raise ValueError(\'Cannot order reference centerline without losing pixels\')\n    return skel, np.asarray(order), closed\n\n\ndef _disk_region(shape, point, radius):\n    yy, xx = np.indices(shape)\n    return ((yy - point[0]) ** 2 + (xx - point[1]) ** 2 <= radius ** 2)\n\n\nclass _Pair(nn.Module):\n    def __init__(self, corridor, a, b, slices):\n        super().__init__()\n        for name, value in [(\'corridor\', corridor), (\'a\', a), (\'b\', b)]:\n            self.register_buffer(name, torch.tensor(value, dtype=torch.float32)[None, None])\n        self.slices = slices\n\n\nclass _Stroke(nn.Module):\n    def __init__(self, reference, config):\n        super().__init__()\n        skel, order, self.closed = _ordered_centerline(reference > config.reference_threshold)\n        self.length = len(order)\n        self.register_buffer(\'skeleton\', torch.tensor(skel, dtype=torch.float32)[None, None])\n        self.register_buffer(\'ys\', torch.tensor(order[:, 0], dtype=torch.long))\n        self.register_buffer(\'xs\', torch.tensor(order[:, 1], dtype=torch.long))\n        distance = distance_transform_edt(~skel)\n        self.register_buffer(\'seeds\', torch.tensor(distance <= config.junction_seed_radius,\n                                                   dtype=torch.float32)[None, None])\n        normal = distance <= max(0.5, config.expected_width / 2)\n        self.register_buffer(\'normal\', torch.tensor(normal, dtype=torch.float32)[None, None])\n        anchors = list(range(0, len(order), config.anchor_spacing))\n        if not self.closed and anchors[-1] != len(order) - 1:\n            anchors.append(len(order) - 1)\n        raw_pairs = []\n        for span in config.pair_spans:\n            for i in range(len(anchors)):\n                j = i + span\n                if not self.closed and j >= len(anchors):\n                    continue\n                if self.closed and span >= len(anchors):\n                    continue\n                begin, end = anchors[i], anchors[j % len(anchors)]\n                path = (order[begin:end + 1] if end > begin else\n                        np.concatenate((order[begin:], order[:end + 1])))\n                if len(path) <= 2 * config.anchor_radius + 2:\n                    continue\n                # Keep anchor patches disjoint, including on tight curves.\n                if np.linalg.norm(order[begin] - order[end]) <= 2 * config.anchor_radius + 1:\n                    continue\n                segment = np.zeros_like(skel)\n                segment[path[:, 0], path[:, 1]] = True\n                corridor = distance_transform_edt(~segment) <= config.corridor_radius\n                a = _disk_region(skel.shape, order[begin], config.anchor_radius) & corridor\n                b = _disk_region(skel.shape, order[end], config.anchor_radius) & corridor\n                yy, xx = np.where(corridor)\n                slices = (slice(yy.min(), yy.max() + 1), slice(xx.min(), xx.max() + 1))\n                raw_pairs.append((corridor[slices], a[slices], b[slices], slices))\n        if len(raw_pairs) > config.max_pairs:\n            ids = np.linspace(0, len(raw_pairs) - 1, config.max_pairs).round().astype(int)\n            raw_pairs = [raw_pairs[i] for i in ids]\n        self.pairs = nn.ModuleList([_Pair(*args) for args in raw_pairs])\n        if not self.pairs:\n            raise ValueError(\'No valid anchor pairs: reduce anchor_spacing/radius for this reference\')\n\n\ndef _ring_offsets(radius):\n    # Sample a digital square perimeter in cyclic order, with no duplicates.\n    return ([( -radius, x) for x in range(-radius, radius)] +\n            [(y, radius) for y in range(-radius, radius)] +\n            [(radius, x) for x in range(radius, -radius, -1)] +\n            [(y, -radius) for y in range(radius, -radius, -1)])\n\n\ndef _crossings(skeleton, radius):\n    h, w = skeleton.shape[-2:]\n    padded = F.pad(skeleton, (radius,) * 4)\n    samples = [padded[..., radius + y:radius + y + h, radius + x:radius + x + w]\n               for y, x in _ring_offsets(radius)]\n    return sum((a - b).abs() for a, b in zip(samples, samples[1:] + samples[:1])) / 2\n\n\ndef soft_occupancy(mask, threshold, temperature):\n    """Approximate absolute level-set occupancy, with exactly 0->0 and 1->1.\n\n    This is NOT per-stroke max normalization. An empty/faint missing stroke is\n    never amplified relative to its own maximum. Shape and absolute ink are\n    separated using shared, configurable thresholds.\n    """\n    tau = torch.as_tensor(threshold, device=mask.device, dtype=mask.dtype)\n    low = torch.sigmoid(-tau / temperature)\n    high = torch.sigmoid((1 - tau) / temperature)\n    return ((torch.sigmoid((mask - tau) / temperature) - low) / (high - low)).clamp(0, 1)\n\n\ndef soft_distance_to_background(occupancy, steps):\n    """Differentiable truncated chamfer distance, in input pixels.\n\n    Fixed iterations; min has the same subgradient qualification as pooling.\n    This estimates the probe scale; it never edits or fills the input mask.\n    """\n    distance = occupancy * steps\n    h, w = occupancy.shape[-2:]\n    for _ in range(steps):\n        padded = F.pad(distance, (1, 1, 1, 1), value=0)\n        neighbors = [distance]\n        for dy, dx in ((-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)):\n            neighbors.append(padded[..., 1+dy:1+dy+h, 1+dx:1+dx+w] + math.hypot(dy, dx))\n        distance = torch.stack(neighbors).amin(0)\n    return distance\n\n\ndef occupancy_ring_arms(occupancy, radius, samples, smoothing):\n    """Count resolved angular transitions on the actual occupancy, not skeleton."""\n    n, _, h, w = occupancy.shape\n    angles = torch.arange(samples, device=occupancy.device, dtype=occupancy.dtype) * (2*math.pi/samples)\n    yy, xx = torch.meshgrid(torch.arange(h, device=occupancy.device, dtype=occupancy.dtype),\n                            torch.arange(w, device=occupancy.device, dtype=occupancy.dtype), indexing=\'ij\')\n    x = xx[None, :, :, None] + radius[:, 0, :, :, None] * angles.cos()\n    y = yy[None, :, :, None] + radius[:, 0, :, :, None] * angles.sin()\n    grid = torch.stack(((x+.5)*2/w-1, (y+.5)*2/h-1), -1).reshape(n, h, w*samples, 2)\n    ring = F.grid_sample(occupancy, grid, align_corners=False).reshape(n, 1, h, w, samples)\n    for _ in range(smoothing):\n        ring = (ring.roll(1, -1) + 2*ring + ring.roll(-1, -1))/4\n    return (ring-ring.roll(1, -1)).abs().sum(-1)/2\n\n\ndef multilevel_junction(mask, seeds, config):\n    """Differentiate shape per absolute level BEFORE counting skeleton arms.\n\n    Levels are stacked on the batch axis. Return a strongest-level response map\n    and level diagnostics. A branch with ink .6 is not counted as 3*.6 arms.\n    """\n    b, _, h, w = mask.shape\n    levels = len(config.junction_thresholds)\n    tau = mask.new_tensor(config.junction_thresholds)[:, None, None, None, None]\n    occupancy = soft_occupancy(mask[None], tau, config.junction_temperature).reshape(levels*b, 1, h, w)\n    skeleton = soft_skeleton(occupancy, config.skeleton_iterations)\n    attached, converged = reconstruct(occupancy, occupancy * seeds, config.attachment_steps)\n    local = torch.stack([F.relu(_crossings(skeleton, r)-2-config.junction_margin).square()\n                         for r in config.junction_radii]).mean(0)\n    local = F.max_pool2d(local, 5, 1, 2)\n    distance = soft_distance_to_background(occupancy, config.junction_distance_steps)\n    radius = config.junction_probe_scale*distance + config.junction_probe_offset\n    arms = occupancy_ring_arms(occupancy, radius, config.junction_angular_samples,\n                               config.junction_angular_smoothing)\n    resolved = F.relu(arms-2-config.junction_ring_margin).square()\n    maps = (resolved * local * skeleton * attached).reshape(levels, b, 1, h, w)\n    # Average evidence instead of selecting a worst pixel from any one threshold.\n    response = maps.mean(0)\n    return response, maps, converged\n\n\nclass StrokeStructureLoss(nn.Module):\n    """Prepare once on fixed aligned reference strokes, then call on live masks.\n\n    reference_strokes: K,H,W. forward: K,H,W or B,K,H,W, floating [0,1].\n    Returned per_stroke values are B,K. loss is an EXTRACTOR structural loss,\n    not a source-image quality score. Reference deficits are returned separately.\n    Closed curves are ordered\n    cyclically; local corridors prevent bypassing a gap via the rest of a ring.\n    No alignment is estimated here. Geometry must already be in input coordinates.\n    """\n    def __init__(self, reference_strokes, config=None):\n        super().__init__()\n        self.config = config or StructureConfig()\n        ref = torch.as_tensor(reference_strokes).detach().float().cpu().numpy()\n        if ref.ndim != 3 or not np.isfinite(ref).all() or (ref < 0).any() or (ref > 1).any():\n            raise ValueError(\'reference_strokes must be finite K,H,W in [0,1]\')\n        self.shape = ref.shape\n        prepared = []\n        for k, stroke in enumerate(ref):\n            try:\n                prepared.append(_Stroke(stroke, self.config))\n            except ValueError as exc:\n                raise ValueError(f\'Reference stroke {k + 1}: {exc}\') from exc\n        if not prepared:\n            raise ValueError(\'At least one reference stroke is required\')\n        self.strokes = nn.ModuleList(prepared)\n\n    def forward(self, masks, return_maps=False):\n        if masks.ndim == 3:\n            masks = masks[None]\n        if masks.ndim != 4 or tuple(masks.shape[1:]) != tuple(self.shape):\n            raise ValueError(f\'masks must have trailing shape {self.shape}\')\n        if not masks.is_floating_point():\n            raise ValueError(\'masks must be floating point\')\n        if not bool(torch.isfinite(masks).all()) or bool((masks < 0).any()) or bool((masks > 1).any()):\n            raise ValueError(\'masks must be finite in [0,1]; do not normalize each stroke\')\n        if next(self.buffers()).device != masks.device:\n            raise ValueError(\'Move the prepared loss to masks.device with .to(device)\')\n        cfg = self.config\n        metrics = {name: [] for name in (\'missing\', \'split\', \'junction\', \'width\')}\n        all_maps, convergence = [], []\n        for k, stroke in enumerate(self.strokes):\n            m = masks[:, k:k + 1].float()\n            coverage = _pool(m, cfg.coverage_radius)[:, 0, stroke.ys, stroke.xs]\n            deficits = F.relu(1 - coverage / cfg.required_ink).square()\n            missing = deficits.mean(-1)\n            local_mass = _pool(m, cfg.width_radius, mean=True)[:, 0, stroke.ys, stroke.xs]\n            expected = _pool(stroke.normal, cfg.width_radius, mean=True)[:, 0, stroke.ys, stroke.xs]\n            width = F.relu(1 - local_mass / (expected * (1 - cfg.width_tolerance) + cfg.eps)).square().mean(-1)\n            split_values, split_maps, pair_converged = [], [], []\n            for pair in stroke.pairs:\n                f = m[..., pair.slices[0], pair.slices[1]] * pair.corridor\n                reached, converged = reconstruct(f, f * pair.a, cfg.connectivity_steps)\n                u = (f * pair.a).flatten(1).amax(1)\n                v = (f * pair.b).flatten(1).amax(1)\n                c = (reached * pair.b).flatten(1).amax(1)\n                strength = torch.minimum(u, v)\n                # clamp avoids the tiny nonzero epsilon penalty on perfect paths.\n                gap = F.relu((strength - c) / strength.clamp_min(cfg.eps)).square()\n                split_values.append(u * v * gap)\n                pair_converged.append(converged)\n                if return_maps:\n                    split_maps.append((pair, F.relu(f - reached)))\n            split = torch.stack(split_values, -1).mean(-1)\n            attached, attach_converged = reconstruct(m, m * stroke.seeds, cfg.attachment_steps)\n            junction_map, junction_levels, junction_converged = multilevel_junction(m, stroke.seeds, cfg)\n            junction = junction_map.flatten(1).sum(-1) / stroke.length\n            for name, value in [(\'missing\', missing), (\'split\', split), (\'junction\', junction), (\'width\', width)]:\n                metrics[name].append(value)\n            convergence.append({\'stroke\': k + 1, \'pairs\': all(pair_converged),\n                                \'attachment\': attach_converged and junction_converged, \'closed\': stroke.closed})\n            if return_maps:\n                missing_map = torch.zeros_like(m)\n                missing_map[:, 0, stroke.ys, stroke.xs] = deficits\n                split_map = torch.zeros_like(m)\n                for pair, area in split_maps:\n                    # Maps are diagnostics, not the differentiable loss path.\n                    with torch.no_grad():\n                        target = split_map[..., pair.slices[0], pair.slices[1]]\n                        target.copy_(torch.maximum(target, area.detach()))\n                all_maps.append({\'mask\': m.detach(), \'centerline\': stroke.skeleton.detach(),\n                                 \'missing\': missing_map.detach(), \'split\': split_map.detach(),\n                                 \'junction\': junction_map.detach(), \'attached\': attached.detach(),\n                                 \'junction_level_scores\': (junction_levels.flatten(2).sum(-1) / stroke.length).detach()})\n        per_stroke = {key: torch.stack(value, -1) for key, value in metrics.items()}\n        extraction = cfg.split_weight * per_stroke[\'split\'] + cfg.junction_weight * per_stroke[\'junction\']\n        reference = cfg.missing_weight * per_stroke[\'missing\'] + cfg.width_weight * per_stroke[\'width\']\n        result = {\'loss\': extraction.mean(), \'reference_loss\': reference.mean(),\n                  \'per_stroke\': {**per_stroke, \'extraction_error\': extraction,\n                                 \'reference_deficit\': reference},\n                  \'source_quality\': {\'assessed\': False,\n                      \'reason\': \'Extracted-mask defects do not identify source-image defects.\'},\n                  \'convergence\': convergence}\n        if return_maps:\n            result[\'maps\'] = all_maps\n        return result\n', 'stroke_model/source_structure.py': '"""Source-image errors after ONE bounded affine registration of the target.\n\nExtractor channel mistakes are diagnostics, not evidence of missing source ink.\nRegistration is CPU evaluation preprocessing, not differentiable guidance.\nPreparedSourceMetric retains image gradients with the registration held fixed.\n"""\nfrom dataclasses import dataclass, asdict\nimport math\nimport numpy as np\nfrom scipy.ndimage import affine_transform, distance_transform_edt\nfrom scipy.optimize import minimize\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom .stroke_structure import (StructureConfig, _Stroke, _pool, reconstruct,\n    soft_occupancy, soft_skeleton, soft_distance_to_background, occupancy_ring_arms)\n\n\n@dataclass(frozen=True)\nclass SourceConfig:\n    translation: float = 12.\n    rotation: float = 15.\n    scale_min: float = .85\n    scale_max: float = 1.2\n    shear: float = .2\n    registration_iterations: int = 45\n    registration_prior: float = .00001\n    extra_radius: float = 2.\n    extra_transition: float = .5\n    junction_tolerance: float = .5\n    junction_position_tolerance: int = 2\n    coverage_radius: int = 2\n    corridor_radius: int = 3\n\n    def __post_init__(self):\n        if any(not math.isfinite(x) for x in asdict(self).values()):\n            raise ValueError(\'Source settings must be finite\')\n        if (self.translation <= 0 or self.rotation <= 0 or self.shear <= 0\n                or not 0 < self.scale_min <= 1 <= self.scale_max or self.scale_min == self.scale_max\n                or self.registration_iterations < 1 or self.registration_prior < 0\n                or self.extra_radius < 0 or self.extra_transition <= 0 or self.junction_tolerance < 0):\n            raise ValueError(\'Invalid source settings\')\n        for key in (\'registration_iterations\',\'junction_position_tolerance\',\'coverage_radius\',\'corridor_radius\'):\n            if not isinstance(getattr(self,key), int) or getattr(self,key)<0:\n                raise ValueError(\'Source iteration counts and radii must be nonnegative integers\')\n\n\ndef affine_warp(array, parameters):\n    tx, ty, angle, sx, sy, shear = parameters\n    angle = np.deg2rad(angle)\n    rotation = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])\n    matrix = rotation @ np.array([[sy, shear], [0, sx]])  # y,x convention\n    inverse = np.linalg.inv(matrix)\n    center = (np.asarray(array.shape)-1)/2\n    offset = center-inverse@(center+np.array([ty,tx]))\n    return affine_transform(array, inverse, offset, order=1, mode=\'constant\')\n\n\ndef register_reference(image, reference, strokes, config=None):\n    cfg = config or SourceConfig()\n    image, reference, strokes = [np.asarray(x, dtype=np.float32) for x in (image,reference,strokes)]\n    image, reference = image.squeeze(), reference.squeeze()\n    if (image.ndim!=2 or reference.shape!=image.shape or strokes.ndim!=3\n            or strokes.shape[1:]!=image.shape or not len(strokes)):\n        raise ValueError(\'Need matching input H,W, full target H,W and target strokes K,H,W\')\n    for value in (image,reference,strokes):\n        if not np.isfinite(value).all() or (value<0).any() or (value>1).any():\n            raise ValueError(\'Source arrays must be finite absolute ink in [0,1]\')\n    if reference.max()<=.5 or any(s.max()<=.5 for s in strokes):\n        raise ValueError(\'Target and all expected target strokes must be nonempty\')\n    def objective(parameters):\n        moved = affine_warp(reference, parameters)\n        dice = 2*(image*moved).sum()/(np.square(image).sum()+np.square(moved).sum()+1e-6)\n        return 1-dice+cfg.registration_prior*sum(v*v for v in parameters[:3])\n    bounds=[(-cfg.translation,cfg.translation),(-cfg.translation,cfg.translation),\n            (-cfg.rotation,cfg.rotation),(cfg.scale_min,cfg.scale_max),\n            (cfg.scale_min,cfg.scale_max),(-cfg.shear,cfg.shear)]\n    fit=minimize(objective,[0,0,0,1,1,0],method=\'Powell\',bounds=bounds,\n                 options={\'maxiter\':cfg.registration_iterations,\'xtol\':.001,\'ftol\':.00001})\n    # An optimizer result may be worse than identity; preserve the best of both.\n    parameters = fit.x if objective(fit.x)<objective([0,0,0,1,1,0]) else np.array([0,0,0,1,1,0])\n    target=affine_warp(reference,parameters)\n    dice=2*(image*target).sum()/(np.square(image).sum()+np.square(target).sum()+1e-6)\n    # Keep reference centerline topology intact; sample the live input into\n    # reference coordinates instead of re-skeletonizing a raster-warped target.\n    return reference, strokes, {\'parameters\':parameters.tolist(),\'dice\':float(dice),\n                            \'optimizer_converged\':bool(fit.success),\'fixed_for_gradients\':True}\n\n\nclass PreparedSourceMetric(nn.Module):\n    def __init__(self, aligned_target, aligned_strokes, config=None, parameters=None):\n        super().__init__()\n        self.config=config or SourceConfig()\n        self.structure_config=StructureConfig(coverage_radius=self.config.coverage_radius,\n            corridor_radius=self.config.corridor_radius, reference_threshold=.5)\n        self.strokes=nn.ModuleList([_Stroke(s,self.structure_config) for s in aligned_strokes])\n        self.register_buffer(\'target\',torch.tensor(aligned_target,dtype=torch.float32)[None,None])\n        distance=distance_transform_edt(np.asarray(aligned_target)<.3)\n        support=1/(1+np.exp(np.clip((distance-self.config.extra_radius)/self.config.extra_transition,-80,80)))\n        # Exact support=1 inside target ink; avoid charging even an identical\n        # image for the sigmoid\'s nonzero interior tail.\n        support=np.minimum(support/(1/(1+np.exp(-self.config.extra_radius/self.config.extra_transition))),1)\n        self.register_buffer(\'support\',torch.tensor(support,dtype=torch.float32)[None,None])\n        pose=parameters if parameters is not None else [0,0,0,1,1,0]\n        source_support=affine_warp(support,pose)\n        self.register_buffer(\'source_support\',torch.tensor(source_support,dtype=torch.float32)[None,None])\n        weights=np.stack([np.exp(-distance_transform_edt(~(s>.15))/8.) for s in aligned_strokes])\n        self.register_buffer(\'junction_weights\',torch.tensor(weights,dtype=torch.float32))\n        h,w=self.target.shape[-2:]\n        tx,ty,angle,sx,sy,shear=parameters if parameters is not None else [0,0,0,1,1,0]\n        angle=np.deg2rad(angle)\n        rotation=np.array([[np.cos(angle),-np.sin(angle)],[np.sin(angle),np.cos(angle)]])\n        matrix=rotation@np.array([[sy,shear],[0,sx]])\n        yy,xx=np.indices((h,w));center=(np.array([h,w])-1)/2\n        positions=matrix@np.stack([yy-center[0],xx-center[1]]).reshape(2,-1)\n        positions+= (center+np.array([ty,tx]))[:,None]\n        grid=np.stack([(positions[1].reshape(h,w)+.5)*2/w-1,\n                       (positions[0].reshape(h,w)+.5)*2/h-1],axis=-1)\n        self.register_buffer(\'sampling_grid\',torch.tensor(grid,dtype=torch.float32)[None])\n\n    def forward(self,image):\n        image=torch.as_tensor(image)\n        if image.ndim==2:image=image[None,None]\n        elif image.ndim==3:image=image[None]\n        if image.ndim!=4 or image.shape[1:]!=self.target.shape[1:]:\n            raise ValueError(\'Image must be H,W, 1,H,W or B,1,H,W matching the prepared target\')\n        if not image.is_floating_point():image=image.float()\n        if image.device!=self.target.device or not bool(torch.isfinite(image).all()) or bool(((image<0)|(image>1)).any()):\n            raise ValueError(\'Image must be finite ink [0,1] on the metric device\')\n        source_image=image\n        image=F.grid_sample(image,self.sampling_grid.to(image).expand(image.shape[0],-1,-1,-1),align_corners=False)\n        cfg=self.structure_config\n        missing,split=[],[]\n        for stroke in self.strokes:\n            coverage=_pool(image,cfg.coverage_radius)[:,0,stroke.ys,stroke.xs]\n            missing.append(F.relu(1-coverage/cfg.required_ink).square().mean(-1))\n            pairs=[]\n            for pair in stroke.pairs:\n                ink=image[...,pair.slices[0],pair.slices[1]]*pair.corridor\n                reached,_=reconstruct(ink,ink*pair.a)\n                u=(ink*pair.a).flatten(1).amax(1)\n                v=(ink*pair.b).flatten(1).amax(1)\n                c=(reached*pair.b).flatten(1).amax(1)\n                pairs.append(u*v*F.relu((torch.minimum(u,v)-c)/torch.minimum(u,v).clamp_min(cfg.eps)).square())\n            split.append(torch.stack(pairs,-1).mean(-1))\n        levels=[]\n        for threshold in cfg.junction_thresholds:\n            occupancy=soft_occupancy(image,threshold,cfg.junction_temperature)\n            expected=soft_occupancy(self.target.to(image),threshold,cfg.junction_temperature).expand_as(image)\n            radius=cfg.junction_probe_scale*soft_distance_to_background(occupancy,cfg.junction_distance_steps)+cfg.junction_probe_offset\n            arms=occupancy_ring_arms(occupancy,radius,cfg.junction_angular_samples,cfg.junction_angular_smoothing)\n            target_arms=occupancy_ring_arms(expected,radius,cfg.junction_angular_samples,cfg.junction_angular_smoothing)\n            allowed=_pool(target_arms,self.config.junction_position_tolerance).clamp_min(2)\n            excess=F.relu(arms-allowed-self.config.junction_tolerance).square()\n            levels.append(excess*soft_skeleton(occupancy,cfg.skeleton_iterations))\n        junction_map=torch.stack(levels).mean(0)\n        junction=torch.stack([(junction_map[:,0]*w).flatten(1).sum(1)/s.length\n                              for w,s in zip(self.junction_weights,self.strokes)],-1)\n        # Inspect EVERY original pixel, including margins outside the registered\n        # canonical view; an affine crop must never hide an added stroke.\n        extra_map=source_image*(1-self.source_support)\n        extra=extra_map.flatten(1).sum(1)/source_image.flatten(1).sum(1).clamp_min(1)\n        return {\'missing\':torch.stack(missing,-1),\'split\':torch.stack(split,-1),\'junction\':junction,\n                \'extra\':extra,\'extra_map\':extra_map,\'junction_map\':junction_map,\'support\':self.source_support,\n                \'registered_image\':image,\'source_image\':source_image}\n', 'stroke_model/target_stroke_plan.py': '"""Observed widths -> canonical strokes -> one glyph-level affine -> edit plan.\n\nScores select edit strength; they are not differentiated through a generator.\nThe built-in geometry scores are explicit uncalibrated engineering proxies, NOT\na reproduction of Li & Ren\'s calligraphy evaluator or extraction confidence.\n"""\nfrom dataclasses import dataclass, asdict\nimport math\nimport numpy as np\nfrom PIL import Image\nfrom scipy.ndimage import distance_transform_edt, binary_dilation, convolve\nfrom skimage.morphology import skeletonize\nimport torch\n\nfrom .source_structure import SourceConfig, register_reference, affine_warp\n\n\n@dataclass(frozen=True)\nclass TargetPlanConfig:\n    mask_threshold: float = .35\n    keep_score: float = 90.\n    full_edit_score: float = 40.\n    maximum_step_blend: float = .8\n    edit_margin: int = 2\n    width_min_samples: int = 6\n\n    def __post_init__(self):\n        if any(not math.isfinite(v) for v in asdict(self).values()):raise ValueError(\'Nonfinite config\')\n        if not 0<self.mask_threshold<1 or not 0<=self.full_edit_score<self.keep_score<=100:\n            raise ValueError(\'Invalid threshold or score range\')\n        if not 0<self.maximum_step_blend<=1 or type(self.edit_margin) is not int or self.edit_margin<0:\n            raise ValueError(\'Invalid blend/margin\')\n        if type(self.width_min_samples) is not int or self.width_min_samples<1:raise ValueError(\'Invalid sample count\')\n\n\ndef _features(mask):\n    binary=mask>.35;path=skeletonize(binary)\n    if not path.any():return dict(present=False)\n    # Undirected 8-neighbour edge length, counting each edge once.\n    length=0.\n    for dy,dx,cost in [(0,1,1.),(1,0,1.),(1,1,2**.5),(1,-1,2**.5)]:\n        a=path[:path.shape[0]-dy or None, max(0,-dx):path.shape[1]-max(0,dx)]\n        b=path[dy:,max(0,dx):path.shape[1]-max(0,-dx)]\n        length+=float((a&b).sum())*cost\n    xy=np.argwhere(path)[:,::-1].astype(float)\n    center=xy.mean(0);cov=np.cov(xy.T) if len(xy)>2 else np.eye(2)\n    values,vectors=np.linalg.eigh(cov);axis=vectors[:,-1]\n    ambiguous=bool(values[-1]<1.15*max(values[0],1e-6))\n    angle=float(np.arctan2(axis[1],axis[0]))\n    # Bend is a dimensionless path/span proxy, not circle-fit curvature.\n    span=max(float(np.linalg.norm(xy.max(0)-xy.min(0))),1.)\n    return dict(present=True,length=length,width=float(2*np.median(distance_transform_edt(binary)[path])),\n        angle=angle,angle_ambiguous=ambiguous,bend=length/span,\n        center=center.tolist(),area=int(binary.sum()))\n\n\ndef evaluate_stroke(observed,target):\n    a,b=_features(observed),_features(target)\n    if not a[\'present\']:\n        return dict(score=0.,source=\'geometry_proxy_v1\',observed=a,target=b,reason=\'absent_observed_stroke\')\n    length_ratio=1. if a[\'length\']<1 and b[\'length\']<1 else a[\'length\']/max(b[\'length\'],1.)\n    width_ratio=a[\'width\']/max(b[\'width\'],1e-6)\n    angle_delta=float((a[\'angle\']-b[\'angle\']+np.pi/2)%np.pi-np.pi/2)\n    angle_used=not (a[\'angle_ambiguous\'] or b[\'angle_ambiguous\'])\n    bend_error=abs(a[\'bend\']-b[\'bend\'])\n    ma,mb=observed>.35,target>.35\n    dice=float(2*(ma&mb).sum()/max(int(ma.sum()+mb.sum()),1))\n    # Width is measured and reported, but NOT forced back to template width.\n    error=(abs(math.log(max(length_ratio,1e-6)))+\n           (abs(angle_delta)/(np.pi/4) if angle_used else 0.)+.5*bend_error+(1-dice))\n    return dict(score=float(100*np.exp(-error)),source=\'geometry_proxy_v1\',observed=a,target=b,\n        length_ratio=length_ratio,width_ratio=width_ratio,angle_delta_radians=angle_delta,\n        angle_used=angle_used,bend_error=bend_error,body_dice=dice)\n\n\ndef measure_widths(observed,config):\n    estimates=[]\n    for mask in observed:\n        binary=mask>config.mask_threshold;path=skeletonize(binary)\n        degree=convolve(path.astype(int),np.ones((3,3),int),mode=\'constant\')-path\n        stable=path&(degree==2)\n        values=distance_transform_edt(binary)[stable]\n        estimate=float(np.median(values)) if len(values)>=config.width_min_samples else None\n        estimates.append(estimate)\n    reliable=[w for w in estimates if w is not None and w>0]\n    if not reliable:raise ValueError(\'No measurable observed stroke widths; defer this glyph\')\n    median=float(np.median(reliable))\n    return [dict(half_width=w if w is not None else median,\n                 source=\'observed_stroke\' if w is not None else \'observed_glyph_median\') for w in estimates]\n\n\ndef _background(rgb,foreground):\n    if foreground.all():raise ValueError(\'No background pixels available\')\n    _,indices=distance_transform_edt(foreground,return_indices=True)\n    return rgb[indices[0],indices[1]]\n\n\nclass TargetImageGuide:\n    def __init__(self,source,target,weights,stroke_regions,report,config):\n        self.source=np.asarray(source,np.float32);self.target=np.asarray(target,np.float32)\n        self.weights=np.asarray(weights,np.float32)\n        self.stroke_regions=stroke_regions;self.report=report;self.config=config\n        self.roi=self.weights>0\n        self.metadata=dict(kind=\'observed_width_aligned_target_image\',config=asdict(config),\n            score_role=\'frozen edit selection and blend strength; not a scalar reward gradient\',\n            final_pixel_replacement=False)\n\n    @torch.no_grad()\n    def project(self,rgb):\n        if rgb.shape!=(1,3,*self.weights.shape):raise ValueError(\'Expected matching RGB sheet\')\n        target=torch.as_tensor(self.target.copy(),device=rgb.device).permute(2,0,1)[None].to(rgb)\n        weight=torch.as_tensor(self.weights.copy(),device=rgb.device)[None,None].to(rgb)\n        return rgb+weight*(target-rgb)\n\n    @torch.no_grad()\n    def diagnostics(self,rgb):\n        target=torch.as_tensor(self.target.copy(),device=rgb.device).permute(2,0,1)[None].to(rgb)\n        error=(rgb-target).square().mean(1)[0]\n        rows=[]\n        for region,item in zip(self.stroke_regions,self.report[\'strokes\']):\n            mask=torch.as_tensor(region.copy(),device=rgb.device)\n            rows.append(dict(glyph=item[\'glyph\'],stroke=item[\'stroke\'],original_score=item[\'score\'],\n                edit_strength=item[\'edit_strength\'],target_rgb_mse=float(error[mask].mean()) if mask.any() else 0.))\n        return dict(strokes=rows,target_rgb_mse=float(error.mean()),correctness_score=False)\n\n\ndef build_target_plan(source,conditions,observed_masks,*,scores=None,config=None,registration_config=None):\n    """Build a full target IMAGE before deriving local, score-controlled edits.\n\n    Optional scores: four lists of 0..100 consistency scores, in stroke-ID order.\n    Omit scores to use the disclosed geometry proxy. Correct extraction/canonical\n    stroke correspondence is assumed; extraction reliability is not predicted.\n    """\n    cfg=config or TargetPlanConfig();source=np.asarray(source.convert(\'RGB\'),np.float32)/255\n    h,w=source.shape[:2]\n    if len(conditions)!=4 or len(observed_masks)!=4 or h!=w or h%2:raise ValueError(\'Expected a 2x2 sheet\')\n    if scores is not None and len(scores)!=4:raise ValueError(\'Need four per-glyph score lists\')\n    target=source.copy();weights=np.zeros((h,w),np.float32);regions=[];rows=[];glyph_reports=[];rendered=[]\n    for gi,(condition,observed) in enumerate(zip(conditions,observed_masks)):\n        strokes=np.asarray(condition[\'strokes\'],np.float32);observed=np.asarray(observed,np.float32)\n        if observed.shape!=strokes.shape or strokes.ndim!=3 or not len(strokes):raise ValueError(\'Stroke masks must correspond\')\n        if any(not np.isfinite(a).all() or a.min()<0 or a.max()>1 for a in (strokes,observed)):\n            raise ValueError(\'Expected finite ink masks in [0,1]\')\n        size=strokes.shape[-2:];side=w//2;x0=(gi%2)*side;y0=(gi//2)*side\n        crop=source[y0:y0+side,x0:x0+side];luma=crop@np.array([.299,.587,.114])\n        widths=measure_widths(observed,cfg)\n        masks=[]\n        for stroke,width in zip(strokes,widths):\n            path=skeletonize(stroke>.35)\n            if not path.any():raise ValueError(\'Empty canonical stroke\')\n            # Render EVERY skeleton, including closed curves, at measured width.\n            masks.append(np.clip(width[\'half_width\']+.5-distance_transform_edt(~path),0,1))\n        masks=np.asarray(masks,np.float32)\n        source_ink=np.asarray(Image.fromarray((1-luma).astype(np.float32)).resize(size[::-1],Image.Resampling.BILINEAR))\n        _,_,registration=register_reference(source_ink,masks.max(0),masks,registration_config or SourceConfig())\n        aligned=np.stack([affine_warp(mask,registration[\'parameters\']) for mask in masks]).clip(0,1)\n        supplied=None if scores is None else scores[gi]\n        if supplied is not None and (len(supplied)!=len(strokes) or any(not math.isfinite(s) or not 0<=s<=100 for s in supplied)):\n            raise ValueError(\'Scores must match stroke IDs and be in [0,100]\')\n        foreground=(1-luma)>.35\n        background=_background(crop,foreground)\n        patch=background.copy()\n        glyph_weights=np.zeros(size,np.float32)\n        for k,(alpha,actual,width) in enumerate(zip(aligned,observed,widths)):\n            evaluation=evaluate_stroke(actual,alpha)\n            score=evaluation[\'score\'] if supplied is None else float(supplied[k])\n            strength=float(np.clip((cfg.keep_score-score)/(cfg.keep_score-cfg.full_edit_score),0,1))\n            region=(alpha>.03)|(actual>.15)\n            if cfg.edit_margin:region=binary_dilation(region,iterations=cfg.edit_margin)\n            glyph_weights=np.maximum(glyph_weights,region*strength*cfg.maximum_step_blend)\n            resized=np.asarray(Image.fromarray(alpha).resize((side,side),Image.Resampling.BICUBIC)).clip(0,1)\n            observation=np.asarray(Image.fromarray(actual).resize((side,side),Image.Resampling.BILINEAR))\n            samples=crop[observation>.5]\n            if not len(samples):samples=crop[foreground]\n            color=np.median(samples,axis=0) if len(samples) else np.zeros(3)\n            patch=patch*(1-resized[...,None])+color*resized[...,None]\n            full=np.zeros((h,w),bool)\n            full[y0:y0+side,x0:x0+side]=np.asarray(Image.fromarray(region).resize((side,side),Image.Resampling.NEAREST))\n            regions.append(full)\n            rows.append(dict(glyph=gi+1,stroke=k+1,score=score,\n                score_source=evaluation[\'source\'] if supplied is None else \'provided_per_stroke_score\',\n                edit_strength=strength,width=width,measurements=evaluation))\n        target[y0:y0+side,x0:x0+side]=patch\n        weights[y0:y0+side,x0:x0+side]=np.asarray(Image.fromarray(glyph_weights).resize((side,side),Image.Resampling.BILINEAR))\n        glyph_reports.append(dict(glyph=gi+1,bbox=[x0,y0,x0+side,y0+side],registration=registration))\n        rendered.append(dict(thick_strokes=masks,aligned_strokes=aligned))\n    report=dict(strokes=rows,glyphs=glyph_reports,target_built_from=\'canonical skeletons + observed widths, then shared glyph affine\',\n        score_calibration=\'uncalibrated geometry proxy unless supplied; not paper reproduction\',\n        width_used_for=\'appearance preservation, not forced to reference thickness\',\n        limitations=[\'Canonical stroke correspondence is assumed correct.\',\'Geometry proxy is not a correctness probability.\',\n            \'Nearest visible background is only a provisional target background, not scene inpainting.\',\n            \'Registration may fail on severe errors; inspect parameters and Dice before interpreting results.\'])\n    return TargetImageGuide(source,target,weights,regions,report,cfg),rendered\n', 'diffusion_model/stroke_projected_diffusion.py': '"""Paired DDIM sampling with direct per-stroke x0 projection, no backward pass.\n\nSupply a loaded SD1.5 img2img pipeline with DDIM scheduler and float32 VAE.\nThis is a sampling experiment, not a trained glyph-condition model.\n"""\nfrom copy import deepcopy\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\n\ndef run_projected_pair(pipe, source, constraints, output, *, prompt=\'\', negative_prompt=\'\',\n                       steps=60,strength=.25,seed=2026,cfg_scale=5.,start_from_noise=False,\n                       save_final_projection=False,progress=None):\n    from PIL import Image\n    out=Path(output);out.mkdir(parents=True,exist_ok=False)\n    if not 0<strength<=1 or steps<1 or (not start_from_noise and int(steps*strength)<1) or any(v%8 for v in source.size):\n        raise ValueError(\'Invalid schedule or image size\')\n    if type(pipe.scheduler).__name__ != \'DDIMScheduler\':\n        raise ValueError(\'This experimental sampler supports DDIM only\')\n    device=next(pipe.unet.parameters()).device\n    rgb=torch.tensor(np.asarray(source.convert(\'RGB\')).copy(),device=device).permute(2,0,1)[None].float()/255\n    constraints.diagnostics(rgb)\n    def encode(x):return pipe.vae.encode(x.float()*2-1).latent_dist.mode()*pipe.vae.config.scaling_factor\n    def decode(z):return (pipe.vae.decode(z.float()/pipe.vae.config.scaling_factor,return_dict=False)[0].float()/2+.5).clamp(0,1)\n    def save(x,path):\n        Image.fromarray(np.rint(x[0].permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(\'uint8\')).save(path)\n    traces={}\n    with torch.no_grad():\n        positive,negative=pipe.encode_prompt(prompt,device,1,True,negative_prompt=negative_prompt)\n        embeddings=torch.cat([negative,positive])\n        pipe.scheduler.set_timesteps(steps,device=device)\n        if start_from_noise:\n            ts=pipe.scheduler.timesteps\n        else:\n            ts,_=pipe.get_timesteps(steps,strength,device)\n        clean=encode(rgb)\n        noise=torch.randn(clean.shape,generator=torch.Generator(\'cpu\').manual_seed(seed)).to(device)\n        initial=noise*pipe.scheduler.init_noise_sigma if start_from_noise else pipe.scheduler.add_noise(clean,noise,ts[:1])\n        roi=torch.as_tensor(constraints.roi.copy(),device=device)[None,None].float()\n        gate=F.adaptive_max_pool2d(roi,initial.shape[-2:])\n        for arm in (\'redenoise_only\',\'stroke_projected\'):\n            scheduler=deepcopy(pipe.scheduler);latent=initial.clone();trace=[]\n            for index,t in enumerate(ts):\n                model_input=scheduler.scale_model_input(torch.cat([latent,latent]),t)\n                u,c=pipe.unet(model_input.to(next(pipe.unet.parameters()).dtype),t,\n                    encoder_hidden_states=embeddings,return_dict=False)[0].chunk(2)\n                base=scheduler.step((u+cfg_scale*(c-u)).float(),t,latent,eta=0.,return_dict=True)\n                latent=base.prev_sample.detach();row=dict(step=index,timestep=int(t))\n                if arm==\'stroke_projected\':\n                    prediction=decode(base.pred_original_sample)\n                    target=constraints.project(prediction)\n                    # Re-encoding is not an exact pixel projection. Record both.\n                    corrected=base.pred_original_sample+gate*(encode(target)-base.pred_original_sample)\n                    previous=int(t)-scheduler.config.num_train_timesteps//scheduler.num_inference_steps\n                    alpha=(scheduler.alphas_cumprod[previous] if previous>=0 else scheduler.final_alpha_cumprod).to(latent)\n                    latent=latent+alpha.sqrt()*(corrected-base.pred_original_sample)\n                    row.update(before=constraints.diagnostics(prediction),pixel_projected=constraints.diagnostics(target),\n                               after_vae=constraints.diagnostics(decode(corrected)))\n                trace.append(row)\n                if progress:progress(arm,index+1,len(ts),row)\n            final=decode(latent);save(final,out/(arm+\'.png\'))\n            if arm==\'stroke_projected\' and save_final_projection:\n                # Explicitly separate the raw diffusion output from the final\n                # image-space enforcement. Never label this as raw generation.\n                save(constraints.project(final),out/\'stroke_projected_final_pixel_projection.png\')\n            traces[arm]=trace\n    source.save(out/\'input.png\')\n    report=dict(steps=steps,actual_steps=len(ts),strength=None if start_from_noise else strength,\n        start_from_noise=start_from_noise,source_content_used=not start_from_noise,seed=seed,cfg_scale=cfg_scale,\n        initial_latent_sha256=hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest(),\n        paired_initial_latent=True,backward_passes=0,additional_training=False,traces=traces,\n        guide=getattr(constraints,\'metadata\',dict(kind=\'per_stroke_ink_constraints\')),\n        final_pixel_projection_saved=save_final_projection,\n        saved_png_ink_tolerance=1/255,\n        limitations=[\'Target construction and extraction correspondence must be checked.\',\n                     \'VAE re-encoding can change the target and leak changes beyond a pixel mask.\',\n                     \'A separately requested final pixel projection is postprocessing, not raw generation.\'])\n    (out/\'report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    return report\n', 'repair_audit/stage1.py': '"""First-stage diagnosis of the existing four-glyph cyclic repair, without training.\n\nReads data (never runtime code) from existing sweep/cyclic ZIPs. Automatic width\nand reference-distance measurements are proxies, not character correctness.\n"""\nfrom copy import deepcopy\nimport csv\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport zipfile\n\nimport numpy as np\nfrom PIL import Image\nfrom scipy.ndimage import binary_dilation, distance_transform_edt, maximum_filter\nfrom skimage.morphology import skeletonize\n\n\ndef file_hash(path):\n    digest = hashlib.sha256()\n    with open(path, \'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef json_write(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n\n\ndef _rgb(z, name):\n    with Image.open(io.BytesIO(z.read(name))) as image:\n        return image.convert(\'RGB\').copy()\n\n\ndef _conditions(z, group):\n    result = []\n    for slot in range(1, 5):\n        member = f\'{group}/source_conditions/condition_{slot}.npz\'\n        with np.load(io.BytesIO(z.read(member)), allow_pickle=False) as arrays:\n            reference = arrays[\'reference\'].astype(np.float32)\n            strokes = arrays[\'strokes\'].astype(np.float32)\n        if (reference.ndim != 3 or reference.shape[0] != 1 or\n                reference.shape[1] != reference.shape[2] or strokes.ndim != 3 or\n                strokes.shape[1:] != reference.shape[1:] or not len(strokes)):\n            raise ValueError(f\'Invalid reference/stroke dimensions: {member}\')\n        for value in (reference, strokes):\n            if not np.isfinite(value).all() or value.min() < 0 or value.max() > 1:\n                raise ValueError(f\'Expected finite [0,1] ink arrays: {member}\')\n        if reference.max() <= .5 or (strokes.max((1, 2)) <= .5).any():\n            raise ValueError(f\'Empty glyph/stroke: {member}\')\n        result.append(dict(reference=reference, strokes=strokes))\n    return result\n\n\ndef read_archive(path, output, selected_case_ids=None, include_controls=True):\n    """Materialize only named PNG/NPZ data, ignoring saved absolute paths/code."""\n    with zipfile.ZipFile(path) as check:\n        if \'archive_metadata.json\' in check.namelist():\n            return read_stage1_archive(path, output, selected_case_ids, include_controls)\n    destination = Path(output)\n    if destination.exists():\n        raise FileExistsError(destination)\n    selected = None if selected_case_ids is None else set(selected_case_ids)\n    with zipfile.ZipFile(path) as z:\n        experiment = json.loads(z.read(\'experiment.json\'))\n        expected = experiment.get(\'checkpoint_sha256\', \'\')\n        if not re.fullmatch(r\'[0-9a-fA-F]{64}\', expected):\n            raise ValueError(\'Archive experiment.json must record the extractor SHA256\')\n        cases = json.loads(z.read(\'cases.json\'))\n        available = {c[\'case_id\'] for c in cases if not c.get(\'control\')}\n        if selected is not None and (not selected or selected - available):\n            raise ValueError(f\'Unknown/empty case selection; available: {sorted(available)}\')\n        chosen = [c for c in cases if not c.get(\'control\') and\n                  (selected is None or c[\'case_id\'] in selected)]\n        if not chosen:\n            raise ValueError(\'No generated cases in this archive\')\n        prepared, controls = [], set()\n        destination.mkdir(parents=True)\n        for item in chosen:\n            group, case_id = item[\'group_id\'], item[\'case_id\']\n            if not re.fullmatch(r\'group_[0-9]+\', group) or not re.fullmatch(r\'[A-Za-z0-9_-]+\', case_id):\n                raise ValueError(\'Invalid archive group/case identifier\')\n            conditions = _conditions(z, group)\n            report = json.loads(z.read(f\'{group}/generation/generation_report.json\'))\n            basename = PurePosixPath(str(item[\'input\']).replace(\'\\\\\', \'/\')).name\n            matches = [p for p in report[\'pages\'] if\n                       PurePosixPath(p[\'input\']).name == basename and not p.get(\'control\')]\n            if len(matches) != 1:\n                raise ValueError(f\'Generated page is not uniquely recorded: {case_id}\')\n            pages = [(case_id, matches[0], \'generated\')]\n            if include_controls and group not in controls:\n                candidates = [p for p in report[\'pages\'] if p.get(\'control\') and p[\'glyph_count\'] == 4]\n                if len(candidates) != 1:\n                    raise ValueError(\'Need one recorded four-glyph normal input control\')\n                pages.append((group + \'_normal_control\', candidates[0], \'normal_control\'))\n                controls.add(group)\n            for identifier, page, kind in pages:\n                if page[\'glyph_count\'] != 4 or page.get(\'safety_filtered\'):\n                    raise ValueError(\'Requires a non-filtered four-glyph page\')\n                member = f"{group}/generation/page_{page[\'page_id\']:03d}.png"\n                source = _rgb(z, member)\n                if source.width != source.height or source.width % 16:\n                    raise ValueError(\'Expected square four-cell image with side divisible by 16\')\n                target = destination / identifier\n                target.mkdir()\n                source.save(target / \'input.png\')\n                for slot, condition in enumerate(conditions, 1):\n                    np.savez_compressed(target / f\'condition_{slot}.npz\', **condition)\n                metadata = dict(case_id=identifier, kind=kind, archive_member=member,\n                    source_sha256=hashlib.sha256(z.read(member)).hexdigest(),\n                    prompt=report.get(\'prompts\', {}).get(\'4\', item.get(\'prompt\', \'\')),\n                    negative_prompt=report.get(\'negative_prompt\', \'\'),\n                    generation_seed=page.get(\'seed\'), generation_strength=page.get(\'strength\'),\n                    checkpoint_sha256=expected.lower())\n                json_write(target / \'input_metadata.json\', metadata)\n                prepared.append(dict(directory=str(target), **metadata))\n    json_write(destination / \'archive_metadata.json\', dict(archive_sha256=file_hash(path),\n        experiment=experiment, selected_cases=prepared))\n    return prepared, experiment\n\n\ndef read_stage1_archive(path, output, selected_case_ids=None, include_controls=True):\n    """Reuse original inputs from an audit ZIP, never the already guided outputs."""\n    destination = Path(output)\n    if destination.exists():\n        raise FileExistsError(destination)\n    with zipfile.ZipFile(path) as z:\n        metadata = json.loads(z.read(\'archive_metadata.json\'))\n        experiment = metadata[\'experiment\']\n        expected = experiment.get(\'checkpoint_sha256\', \'\')\n        if not re.fullmatch(r\'[0-9a-fA-F]{64}\', expected):\n            raise ValueError(\'Missing original extractor SHA256\')\n        recorded = metadata[\'selected_cases\']\n        available = {c[\'case_id\'] for c in recorded if c[\'kind\'] == \'generated\'}\n        selected = available if selected_case_ids is None else set(selected_case_ids)\n        if not selected or selected - available:\n            raise ValueError(f\'Unknown/empty audit selection; available: {sorted(available)}\')\n        chosen = [c for c in recorded if c[\'case_id\'] in selected]\n        groups = {re.fullmatch(r\'(group_[0-9]+)_page_[0-9]+\', c[\'case_id\']).group(1)\n                  for c in chosen if re.fullmatch(r\'(group_[0-9]+)_page_[0-9]+\', c[\'case_id\'])}\n        if len(groups) == 0:\n            raise ValueError(\'Unsupported audit case identifiers\')\n        if include_controls:\n            controls = {c[\'case_id\']: c for c in recorded if c[\'kind\'] == \'normal_control\'}\n            for group in sorted(groups):\n                name = group + \'_normal_control\'\n                if name not in controls:\n                    raise ValueError(\'Audit ZIP lacks the matching normal control\')\n                chosen.append(controls[name])\n        prepared = []\n        destination.mkdir(parents=True)\n        for entry in chosen:\n            identifier = entry[\'case_id\']\n            if not re.fullmatch(r\'group_[0-9]+_(page_[0-9]+|normal_control)\', identifier):\n                raise ValueError(\'Invalid audit identifier\')\n            target = destination / identifier\n            target.mkdir()\n            image = _rgb(z, identifier + \'/input.png\')\n            if image.width != image.height or image.width % 16:\n                raise ValueError(\'Invalid input sheet dimensions\')\n            image.save(target / \'input.png\')\n            # Same validation as the original four source conditions.\n            for slot in range(1, 5):\n                with np.load(io.BytesIO(z.read(f\'{identifier}/condition_{slot}.npz\')), allow_pickle=False) as a:\n                    ref, strokes = a[\'reference\'].astype(np.float32), a[\'strokes\'].astype(np.float32)\n                if (ref.ndim != 3 or ref.shape[0] != 1 or ref.shape[1] != ref.shape[2] or\n                        strokes.ndim != 3 or not len(strokes) or strokes.shape[1:] != ref.shape[1:]):\n                    raise ValueError(\'Invalid condition shape\')\n                if any(not np.isfinite(v).all() or v.min() < 0 or v.max() > 1 for v in (ref, strokes)):\n                    raise ValueError(\'Invalid condition values\')\n                if ref.max() <= .5 or (strokes.max((1, 2)) <= .5).any():\n                    raise ValueError(\'Empty condition\')\n                np.savez_compressed(target/f\'condition_{slot}.npz\', reference=ref, strokes=strokes)\n            row = dict(entry, directory=str(target), checkpoint_sha256=expected.lower(),\n                       reused_stage1_original=True)\n            json_write(target/\'input_metadata.json\', row)\n            prepared.append(row)\n    json_write(destination/\'archive_metadata.json\', dict(archive_sha256=file_hash(path),\n        experiment=experiment, selected_cases=prepared, original_input_reused=True))\n    return prepared, experiment\n\n\ndef load_conditions(directory):\n    result = []\n    for slot in range(1, 5):\n        with np.load(Path(directory) / f\'condition_{slot}.npz\', allow_pickle=False) as a:\n            result.append(dict(reference=a[\'reference\'].copy(), strokes=a[\'strokes\'].copy()))\n    return result\n\n\ndef fit_records(source, conditions, extractor, output, *, fit=True):\n    """Same fitting parameters as the existing cyclic notebook."""\n    import torch\n    from torch.nn import functional as F\n    from stroke_model.stroke_reference_fit import fit_reference_glyph, ReferenceFitConfig\n    from diffusion_model.fitted_candidate_diffusion import image_tensor\n    directory = Path(output)\n    directory.mkdir(parents=True, exist_ok=True)\n    device = next(extractor.parameters()).device\n    rgb = image_tensor(source, device)\n    ink = 1 - (rgb * rgb.new_tensor([.299, .587, .114])[None, :, None, None]).sum(1, keepdim=True)\n    side = source.width // 2\n    records = []\n    for i, condition in enumerate(conditions):\n        ref, strokes = condition[\'reference\'], condition[\'strokes\']\n        x, y = (i % 2) * side, (i // 2) * side\n        crop = F.interpolate(ink[..., y:y+side, x:x+side], ref.shape[-2:], mode=\'bilinear\', align_corners=False)\n        with torch.no_grad():\n            prediction = extractor(crop, torch.as_tensor(ref, device=device)[None],\n                                   torch.as_tensor(strokes, device=device)[None])\n        masks = prediction[\'masks\'][0].cpu().numpy()\n        if not fit:\n            np.savez_compressed(directory / f\'slot_{i+1}.npz\', **condition,\n                extracted_masks=masks, probabilities=prediction[\'probabilities\'][0].cpu().numpy())\n            continue\n        hints = prediction.get(\'hints\', torch.as_tensor(strokes, device=device)[None])[0].cpu().numpy()\n        fitted = fit_reference_glyph(strokes, masks, hints=hints, glyph_reference=ref,\n            config=ReferenceFitConfig(max_displacement=16., smoothness=12., shape_prior=.15, max_junction_shift=2.))\n        record = dict(reference=ref, strokes=strokes, fitted=fitted[\'strokes\'], extracted_masks=masks,\n                      fit_metadata=[r[\'metadata\'] for r in fitted[\'fits\']])\n        records.append(record)\n        np.savez_compressed(directory / f\'slot_{i+1}.npz\', **condition,\n            extracted_masks=masks, fitted_strokes=fitted[\'strokes\'],\n            probabilities=prediction[\'probabilities\'][0].cpu().numpy())\n        json_write(directory / f\'slot_{i+1}_fit.json\', record[\'fit_metadata\'])\n    return records\n\n\ndef ink_array(image):\n    a = np.asarray(image, np.float32) / 255\n    return 1 - (a * np.array([.299, .587, .114], np.float32)).sum(-1)\n\n\ndef fixed_support(source, controller):\n    """Freeze source-based diagnostic support before comparing ANY output arm."""\n    initial_edit = np.maximum(controller.erase.cpu().numpy()[0, 0], controller.restore.cpu().numpy()[0, 0]) > .01\n    preserve = ~binary_dilation(initial_edit, iterations=3)\n    source_ink = ink_array(source)\n    healthy = (source_ink > .5) & preserve\n    center = skeletonize(source_ink > .5) & healthy\n    return dict(healthy=healthy, center=center, preserve=preserve, initial_edit=initial_edit)\n\n\ndef measure(source, result, reference, support, threshold=.5):\n    """Fixed-coordinate width proxy; movement/thresholds confound it. No OCR claim."""\n    original, output, ref = map(ink_array, (source, result, reference))\n    healthy, center = support[\'healthy\'], support[\'center\']\n    source_radius = distance_transform_edt(original > threshold)[center]\n    output_radius = distance_transform_edt(output > threshold)[center]\n    ratio = float(output_radius.mean() / max(source_radius.mean(), 1e-8)) if len(source_radius) else None\n    union = (original > .2) | (ref > .2)\n    def average(values, mask):\n        return float(values[mask].mean()) if mask.any() else None\n    return dict(threshold=threshold, fixed_center_samples=int(center.sum()),\n        healthy_ink_pixels=int(healthy.sum()),\n        fixed_center_width_ratio_proxy=ratio,\n        healthy_ink_drop=average(np.maximum(original-output, 0), healthy),\n        healthy_ink_retention=average((output > threshold).astype(float), healthy),\n        healthy_ink_mse=average((original-output)**2, healthy),\n        reference_distance_on_fixed_union=average((output-ref)**2, union),\n        source_reference_distance_on_fixed_union=average((original-ref)**2, union))\n\n\ndef reference_sheet(conditions, size):\n    result = Image.new(\'RGB\', (size, size), \'white\')\n    half = size // 2\n    for i, c in enumerate(conditions):\n        glyph = Image.fromarray(np.rint((1-c[\'reference\'][0])*255).astype(np.uint8)).convert(\'RGB\')\n        result.paste(glyph.resize((half, half), Image.Resampling.BICUBIC), ((i % 2)*half, (i//2)*half))\n    return result\n\n\ndef width_probe_conditions(conditions):\n    """Optional +1 model-pixel dilation stress test, NOT a validated new font."""\n    return [dict(reference=maximum_filter(c[\'reference\'], size=(1, 3, 3)),\n                 strokes=maximum_filter(c[\'strokes\'], size=(1, 3, 3))) for c in conditions]\n\n\ndef run_case(pipe, extractor, case, *, steps=60, strength=.25, seed=2026,\n             cfg_scale=5., width_probe=False, progress=None,\n             boundary_guard=False, boundary_config=None, observed_width=False, width_config=None,\n             resume=False, on_arm_complete=None):\n    import torch\n    from diffusion_model.fitted_candidate_diffusion import image_tensor, optimize_clean\n    from stroke_model.restore_settle_erase import RestoreSettleErase\n    if steps < 4 or not 0 < strength < 1 or int(steps*strength) < 4:\n        raise ValueError(\'Need at least four actual repair steps\')\n    directory = Path(case[\'directory\'])\n    if observed_width and not boundary_guard:\n        raise ValueError(\'Observed-width comparison requires boundary_guard=True\')\n    out = directory / \'audit\'\n    out.mkdir(exist_ok=resume)\n    with Image.open(directory / \'input.png\') as im:\n        source = im.convert(\'RGB\').copy()\n    conditions = load_conditions(directory)\n    records = None\n    if resume:\n        from .resume import load_records, complete_arm, archive_partial, rgb_hash\n        if (out/\'input.png\').exists() and rgb_hash(out/\'input.png\') != rgb_hash(directory/\'input.png\'):\n            raise ValueError(\'Cached audit input differs from requested source\')\n        records = load_records(out, conditions)\n    if records is None:\n        records = fit_records(source, conditions, extractor, out / \'extraction_input\')\n    source_rgb = image_tensor(source, next(extractor.parameters()).device)\n    controller = RestoreSettleErase(source_rgb, extractor, records)\n    with torch.no_grad():\n        controller.refresh_erase(source_rgb)  # Diagnostic only; first restore step clears erasure.\n    support = fixed_support(source, controller)\n    np.savez_compressed(out / \'fixed_diagnostic_regions.npz\', **support)\n    # Reset the controller: diagnostic setup must not change the repair experiment.\n    controller = RestoreSettleErase(source_rgb, extractor, records)\n    reference = reference_sheet(conditions, source.width)\n    reference.save(out / \'reference_diagnostic_only.png\')\n    def decode(z):\n        return pipe.vae.decode(z.float()/pipe.vae.config.scaling_factor, return_dict=False)[0].float()/2+.5\n    def encode(rgb):\n        return pipe.vae.encode(rgb.float()*2-1).latent_dist.mode()*pipe.vae.config.scaling_factor\n    def save_tensor(rgb, path):\n        a = rgb[0].detach().permute(1, 2, 0).clamp(0, 1).cpu().numpy()\n        Image.fromarray(np.rint(a*255).astype(np.uint8)).save(path)\n    # Same frozen float32 VAE and posterior-mode encoding for every compared arm.\n    # This is deliberately recorded: legacy cyclic used prepare_latents\' sampled posterior.\n    with torch.no_grad():\n        clean_input = encode(source_rgb)\n        save_tensor(decode(clean_input), out / \'vae_roundtrip.png\')\n        positive, negative = pipe.encode_prompt(case[\'prompt\'], \'cuda\', 1, True,\n                                               negative_prompt=case[\'negative_prompt\'])\n        embeddings = torch.cat([negative, positive])\n        pipe.scheduler.set_timesteps(steps, device=\'cuda\')\n        timesteps, _ = pipe.get_timesteps(steps, strength, \'cuda\')\n        noise = torch.randn(clean_input.shape, generator=torch.Generator(\'cpu\').manual_seed(seed),\n                            dtype=torch.float32).to(clean_input.device)\n        initial = pipe.scheduler.add_noise(clean_input, noise, timesteps[:1])\n    initial_hash = hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest()\n    legacy_initial_unrecorded = False\n    if resume:\n        signature = dict(steps=steps, strength=strength, seed=seed, cfg_scale=cfg_scale,\n            boundary_config=boundary_config, width_config=width_config,\n            input_rgb_sha256=rgb_hash(directory/\'input.png\'))\n        cache = out/\'resume_state.json\'\n        if cache.exists():\n            previous_state = json.loads(cache.read_text(encoding=\'utf-8\'))\n            if previous_state[\'signature\'] != signature:\n                raise ValueError(\'Resume settings/input differ from saved state\')\n            with np.load(out/\'resume_initial_latent.npz\', allow_pickle=False) as a:\n                initial = torch.as_tensor(a[\'initial\'].copy(), device=clean_input.device)\n            initial_hash = hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest()\n            if initial_hash != previous_state[\'initial_latent_sha256\']:\n                raise ValueError(\'Cached initial latent is damaged\')\n            legacy_initial_unrecorded = previous_state[\'legacy_initial_unrecorded\']\n        else:\n            old_report = out/\'report.json\'\n            if old_report.exists():\n                expected = json.loads(old_report.read_text(encoding=\'utf-8\'))[\'configuration\'][\'initial_latent_sha256\']\n                if expected != initial_hash:\n                    raise ValueError(\'Historical initial latent differs. Completed cases can be reused, but partial cases need matching runtime or a fresh comparison.\')\n            else:\n                legacy_initial_unrecorded = any(complete_arm(out, a, [int(t) for t in timesteps])\n                                                for a in (\'redenoise_only\',\'cyclic_guided\',\'boundary_guarded\',\'observed_width\'))\n            np.savez_compressed(out/\'resume_initial_latent.npz\', initial=initial.cpu().numpy())\n            json_write(cache, dict(signature=signature, initial_latent_sha256=initial_hash,\n                legacy_initial_unrecorded=legacy_initial_unrecorded))\n    # Expose reconstruction before the actual rerun so failures still retain evidence.\n    source.save(out / \'input.png\')\n    traces = {}\n    arms = [(\'redenoise_only\', None), (\'cyclic_guided\', controller)]\n    if boundary_guard:\n        from stroke_model.boundary_guarded_repair import BoundaryGuardConfig, BoundaryGuardedRepair\n        guard_config = BoundaryGuardConfig(**(boundary_config or {}))\n        guarded = BoundaryGuardedRepair(source_rgb, extractor, records, boundary_config=guard_config)\n        with torch.no_grad():\n            guarded.refresh_erase(source_rgb)\n        np.savez_compressed(out/\'boundary_initial_maps.npz\',\n            erase=guarded.erase.cpu().numpy(), restore=guarded.restore.cpu().numpy(),\n            preserve=guarded.preserve.cpu().numpy())\n        json_write(out/\'boundary_initial_diagnostics.json\', guarded.boundary_history)\n        guarded = BoundaryGuardedRepair(source_rgb, extractor, records, boundary_config=guard_config)\n        arms.append((\'boundary_guarded\', guarded))\n    if observed_width:\n        from stroke_model.observed_width_repair import ObservedWidthConfig, ObservedWidthRepair\n        def calibrated_controller():\n            return ObservedWidthRepair(source_rgb, extractor, records,\n                boundary_config=guard_config, width_config=ObservedWidthConfig(**(width_config or {})))\n        calibrated = calibrated_controller()\n        with torch.no_grad():\n            calibrated.refresh_erase(source_rgb)\n        np.savez_compressed(out/\'observed_width_initial_maps.npz\',\n            erase=calibrated.erase.cpu().numpy(), restore=calibrated.restore.cpu().numpy(),\n            preserve=calibrated.preserve.cpu().numpy())\n        json_write(out/\'observed_width_calibration.json\', calibrated.metadata)\n        json_write(out/\'observed_width_initial_diagnostics.json\', calibrated.boundary_history)\n        arms.append((\'observed_width\', calibrated_controller()))\n    if width_probe:\n        alternate = width_probe_conditions(conditions)\n        alt_records = fit_records(source, alternate, extractor, out / \'width_probe_input\')\n        arms.append((\'reference_width_probe\', RestoreSettleErase(source_rgb, extractor, alt_records)))\n    for arm, objective in arms:\n        if resume and complete_arm(out, arm, [int(t) for t in timesteps]):\n            traces[arm] = json.loads((out/(arm+\'_trace.json\')).read_text(encoding=\'utf-8\'))\n            if progress:\n                progress(case[\'case_id\'], arm, len(timesteps), len(timesteps),\n                         dict(phase=\'cached\', accepted_inner_steps=0))\n            continue\n        if resume:\n            archive_partial(out, arm)\n        scheduler = deepcopy(pipe.scheduler)\n        latent = initial.clone()\n        trace = []\n        frames = out / arm\n        frames.mkdir()\n        for index, t in enumerate(timesteps):\n            with torch.no_grad():\n                model_input = scheduler.scale_model_input(torch.cat([latent, latent]), t)\n                u, c = pipe.unet(model_input.to(dtype=next(pipe.unet.parameters()).dtype), t,\n                    encoder_hidden_states=embeddings, return_dict=False)[0].chunk(2)\n                base = scheduler.step((u+cfg_scale*(c-u)).float(), t, latent, eta=0., return_dict=True)\n            clean, latent = base.pred_original_sample.detach(), base.prev_sample.detach()\n            row = dict(step=index, timestep=int(t), phase=\'baseline\', accepted_inner_steps=0)\n            if objective is not None:\n                with torch.no_grad():\n                    row.update(objective.begin_step(decode(clean), index))\n                if row[\'phase\'] == \'restore\':\n                    clean, projection = objective.project_clean(clean, decode, encode)\n                    row.update(projection)\n                clean, optimization = optimize_clean(clean, decode, objective,\n                    iterations=2 if row[\'phase\'] in (\'restore\', \'settle\') else 4,\n                    step_ratio=.008, budget_ratio=.06, backtracks=5)\n                row.update(optimization)\n                with torch.no_grad():\n                    objective.finish_step(decode(clean), row)\n                previous = int(t) - scheduler.config.num_train_timesteps // scheduler.num_inference_steps\n                alpha = (scheduler.alphas_cumprod[previous] if previous >= 0 else scheduler.final_alpha_cumprod).to(latent)\n                latent = latent + alpha.sqrt() * (clean-base.pred_original_sample)\n                for name in (\'erase\', \'restore\', \'preserve\'):\n                    value = getattr(objective, name)[0, 0].detach().cpu().numpy()\n                    Image.fromarray(np.rint(value.clip(0, 1)*255).astype(np.uint8)).save(frames / f\'{name}_{index:03d}.png\')\n            with torch.no_grad():\n                save_tensor(decode(clean), frames / f\'clean_prediction_{index:03d}.png\')\n            latent = latent.detach()\n            trace.append(row)\n            if progress:\n                progress(case[\'case_id\'], arm, index+1, len(timesteps), row)\n        with torch.no_grad():\n            save_tensor(decode(latent), out / (arm+\'.png\'))\n        traces[arm] = trace\n        json_write(out / (arm+\'_trace.json\'), trace)\n        if objective is not None and hasattr(objective, \'boundary_history\'):\n            json_write(out/(arm+\'_map_history.json\'), objective.boundary_history)\n        if on_arm_complete:\n            on_arm_complete(case[\'case_id\'], arm)\n    results = {\'input\': source, \'vae_roundtrip\': Image.open(out / \'vae_roundtrip.png\').convert(\'RGB\')}\n    for arm, _ in arms:\n        results[arm] = Image.open(out / (arm+\'.png\')).convert(\'RGB\')\n    metric_rows = []\n    for arm, result in results.items():\n        # Save final extraction for VISUAL review; masks are not quality probabilities.\n        if arm != \'input\':\n            fit_records(result, conditions, extractor, out / (\'extraction_\'+arm), fit=False)\n        for slot in range(4):\n            half = source.width // 2\n            x, y = slot % 2 * half, slot // 2 * half\n            box = (x, y, x+half, y+half)\n            local = {k: v[y:y+half, x:x+half] for k, v in support.items()}\n            for threshold in (.35, .5, .65):\n                values = measure(source.crop(box), result.crop(box), reference.crop(box), local, threshold)\n                if boundary_guard:\n                    # Include all source foreground, including tips excluded by\n                    # the original automatic healthy mask. This still isn\'t GT.\n                    ink = ink_array(source.crop(box))\n                    all_source = dict(healthy=ink > .5, center=skeletonize(ink > .5))\n                    full = measure(source.crop(box), result.crop(box), reference.crop(box), all_source, threshold)\n                    values.update({\'all_source_\'+k:full[k] for k in\n                        (\'fixed_center_width_ratio_proxy\',\'healthy_ink_drop\',\'healthy_ink_retention\',\'healthy_ink_mse\')})\n                metric_rows.append(dict(case_id=case[\'case_id\'], kind=case[\'kind\'], slot=slot+1, arm=arm, **values))\n    json_write(out / \'report.json\', dict(case=case, metrics=metric_rows, traces=traces,\n        configuration=dict(steps=steps, actual_steps=len(timesteps), timesteps=[int(t) for t in timesteps],\n            strength=strength, seed=seed, cfg_scale=cfg_scale, vae_dtype=\'float32\',\n            vae_posterior=\'mode_shared_by_all_arms\' if not legacy_initial_unrecorded else \'legacy_initial_not_recorded\',\n            paired_initial_latent=not legacy_initial_unrecorded,\n            resume=resume, legacy_initial_unrecorded=legacy_initial_unrecorded,\n            initial_latent_sha256=hashlib.sha256(initial.cpu().numpy().tobytes()).hexdigest(),\n            width_probe=width_probe, original_repair_parameters=True,\n            boundary_guard=boundary_guard, boundary_config=boundary_config if boundary_guard else None,\n            observed_width=observed_width, width_config=width_config if observed_width else None,\n            legacy_difference=\'Posterior mode, not legacy stochastic VAE encoding\'),\n        limitations=[\'Widths are fixed-source-center distance proxies; movement is a confound.\',\n            \'Healthy masks are heuristic and require human review.\',\n            \'Reference similarity is not correctness and cannot prove template copying.\',\n            \'No automatic pass/fail or independent text accuracy claim.\',\n            \'Width probe is synthetic dilation and may be outside extractor training distribution.\']))\n    make_figures(out, source, reference, results, support)\n    return metric_rows\n\n\ndef make_figures(out, source, reference, results, support):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    labels = list(results)\n    fig, axes = plt.subplots(1, len(labels)+1, figsize=(4*(len(labels)+1), 4), squeeze=False)\n    for ax, (name, image) in zip(axes.flat, [(\'reference (diagnostic)\', reference), *results.items()]):\n        ax.imshow(image); ax.set_title(name); ax.axis(\'off\')\n    fig.tight_layout(); fig.savefig(out / \'comparison.png\', dpi=120); plt.close(fig)\n    fig, axes = plt.subplots(1, 3, figsize=(12, 4))\n    for ax, key in zip(axes, (\'healthy\', \'center\', \'initial_edit\')):\n        ax.imshow(source); ax.imshow(np.ma.masked_where(~support[key], support[key]), cmap=\'autumn\', alpha=.6)\n        ax.set_title(key+\' (heuristic)\'); ax.axis(\'off\')\n    fig.tight_layout(); fig.savefig(out / \'diagnostic_regions.png\', dpi=120); plt.close(fig)\n    for slot in range(1, 5):\n        with np.load(out / \'extraction_input\' / f\'slot_{slot}.npz\') as a:\n            count = len(a[\'extracted_masks\'])\n        fig, axes = plt.subplots(len(results), count+2,\n            figsize=(2.2*(count+2), 2.3*len(results)), squeeze=False)\n        half=source.width//2; x=(slot-1)%2*half; y=(slot-1)//2*half\n        for row, (arm, image) in enumerate(results.items()):\n            with np.load(out / (\'extraction_\'+arm) / f\'slot_{slot}.npz\') as a:\n                images = [(\'reference\', 1-a[\'reference\'][0]),\n                          (arm, np.asarray(image.crop((x,y,x+half,y+half))))] + [\n                    (f\'stroke {i+1}\', 1-m) for i,m in enumerate(a[\'extracted_masks\'])]\n            for ax, (name, value) in zip(axes[row], images):\n                ax.imshow(value, cmap=\'gray\', vmin=0, vmax=1) if value.ndim==2 else ax.imshow(value)\n                ax.set_title(name); ax.axis(\'off\')\n        fig.tight_layout(); fig.savefig(out / f\'extraction_slot_{slot}.png\', dpi=110); plt.close(fig)\n\n\ndef write_tables(root, cases, metrics, failures):\n    root = Path(root)\n    if metrics:\n        with (root / \'automatic_metrics.csv\').open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(metrics[0]))\n            writer.writeheader(); writer.writerows(metrics)\n    fields = [\'case_id\', \'kind\', \'slot\', \'arm\', \'target_character\', \'input_error_type\',\n              \'extraction_ok\', \'error_reduced\', \'thinner\', \'new_error\', \'style_changed\', \'notes\']\n    with (root / \'human_review.csv\').open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        for case in cases:\n            arms = sorted({r[\'arm\'] for r in metrics if r[\'case_id\'] == case[\'case_id\']})\n            for slot in range(1, 5):\n                for arm in arms:\n                    writer.writerow(dict(case_id=case[\'case_id\'], kind=case[\'kind\'], slot=slot, arm=arm))\n    json_write(root / \'failures.json\', failures)\n', 'repair_audit/target_guided_repair.py': '"""Connect the existing 2x2 generator/extractor to an aligned target-image guide.\n\nThe CLI prepares and inspects targets on CPU. The Python repair_generated_sheet\nAPI uses an already loaded pipeline/extractor for actual img2img re-denoising.\n"""\nimport argparse\nimport io\nimport json\nfrom pathlib import Path\nimport zipfile\nimport numpy as np\nfrom PIL import Image, ImageDraw\nimport torch\nfrom torch.nn import functional as F\n\nfrom stroke_model.target_stroke_plan import build_target_plan\nfrom .stage1 import read_archive, load_conditions, json_write\n\n\ndef extract_sheet_masks(source,conditions,extractor):\n    device=next(extractor.parameters(),torch.empty(0)).device\n    rgb=torch.as_tensor(np.asarray(source.convert(\'RGB\')).copy(),device=device).permute(2,0,1)[None].float()/255\n    ink=1-(rgb*rgb.new_tensor([.299,.587,.114])[None,:,None,None]).sum(1,keepdim=True)\n    side=source.width//2;result=[]\n    extractor.eval().requires_grad_(False)\n    with torch.no_grad():\n        for gi,condition in enumerate(conditions):\n            x0=(gi%2)*side;y0=(gi//2)*side\n            ref=torch.as_tensor(condition[\'reference\'],device=device)[None].float()\n            strokes=torch.as_tensor(condition[\'strokes\'],device=device)[None].float()\n            crop=F.interpolate(ink[...,y0:y0+side,x0:x0+side],ref.shape[-2:],mode=\'bilinear\',align_corners=False)\n            result.append(extractor(crop,ref,strokes)[\'masks\'][0].float().cpu().numpy())\n    return result\n\n\ndef save_plan(guide,rendered,output):\n    out=Path(output);out.mkdir(parents=True,exist_ok=False)\n    def image(array):return Image.fromarray(np.rint(array*255).clip(0,255).astype(\'uint8\'))\n    image(guide.source).save(out/\'input.png\');image(guide.target).save(out/\'aligned_target.png\')\n    image(guide.weights).save(out/\'edit_strength.png\')\n    preview=guide.project(torch.tensor(guide.source.copy()).permute(2,0,1)[None])\n    image(preview[0].permute(1,2,0).numpy()).save(out/\'cpu_blend_diagnostic_only.png\')\n    for gi,item in enumerate(rendered,1):np.savez_compressed(out/f\'glyph_{gi}_target_strokes.npz\',**item)\n    np.savez_compressed(out/\'guide.npz\',target=guide.target,weights=guide.weights,source=guide.source)\n    json_write(out/\'plan.json\',dict(**guide.report,guidance=guide.metadata,diffusion_run=False))\n    names=[(\'generated input\',\'input.png\'),(\'width-rendered + aligned target\',\'aligned_target.png\'),\n           (\'score-controlled edit strength\',\'edit_strength.png\'),(\'CPU blend: NOT diffusion\',\'cpu_blend_diagnostic_only.png\')]\n    cell=288;canvas=Image.new(\'RGB\',(cell*4,cell+35),\'white\');draw=ImageDraw.Draw(canvas)\n    for i,(title,name) in enumerate(names):\n        draw.text((i*cell+5,9),title,fill=\'black\')\n        with Image.open(out/name) as im:canvas.paste(im.convert(\'RGB\').resize((cell,cell)),(i*cell,35))\n    canvas.save(out/\'comparison.png\')\n\n\ndef repair_generated_sheet(pipe,extractor,source,conditions,output,*,cached_masks=None,scores=None,\n                           config=None,registration_config=None,steps=60,strength=.25,seed=2026,\n                           prompt=\'\',negative_prompt=\'\',progress=None):\n    """Call directly after existing generate_cases / generation of a 2x2 sheet."""\n    from diffusion_model.stroke_projected_diffusion import run_projected_pair\n    masks=cached_masks if cached_masks is not None else extract_sheet_masks(source,conditions,extractor)\n    guide,rendered=build_target_plan(source,conditions,masks,scores=scores,config=config,\n                                    registration_config=registration_config)\n    root=Path(output);root.mkdir(parents=True,exist_ok=False)\n    save_plan(guide,rendered,root/\'target_plan\')\n    report=run_projected_pair(pipe,source,guide,root/\'reverse\',steps=steps,strength=strength,seed=seed,\n        prompt=prompt,negative_prompt=negative_prompt,start_from_noise=False,save_final_projection=False,progress=progress)\n    json_write(root/\'pipeline.json\',dict(stages=[\'existing_2x2_generation\',\'stroke_extraction\',\n        \'observed_width_rendering\',\'shared_glyph_affine\',\'score_to_edit_plan\',\'source_initialized_rediffusion\'],\n        score_source=\'provided\' if scores is not None else \'uncalibrated_geometry_proxy_v1\',\n        new_training=False,source_content_used=report[\'source_content_used\'],final_pixel_replacement=False,\n        plan=\'target_plan/plan.json\',diffusion=\'reverse/report.json\'))\n    return guide,report\n\n\ndef _cached_masks(z,case,source,conditions):\n    prefix=case[\'case_id\'];result=[]\n    for slot,condition in enumerate(conditions,1):\n        name=f\'{prefix}/audit/extraction_input/slot_{slot}.npz\'\n        if name not in z.namelist():return None\n        with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as a:\n            for key in [\'reference\',\'strokes\']:\n                if key in a and not np.array_equal(a[key],condition[key]):raise ValueError(\'Cached conditions differ\')\n            result.append(a[\'extracted_masks\'].copy())\n    with Image.open(io.BytesIO(z.read(prefix+\'/input.png\'))) as original:\n        if not np.array_equal(np.asarray(original.convert(\'RGB\')),np.asarray(source)):\n            raise ValueError(\'Cached extraction uses different source pixels\')\n    return result\n\n\ndef prepare_archive(archive,output,*,case_ids=None,observations_archive=None,scores=None,config=None):\n    root=Path(output)\n    cases,_=read_archive(archive,root,case_ids,include_controls=True)\n    results=[]\n    with zipfile.ZipFile(archive) as z:\n        for case in cases:\n            directory=Path(case[\'directory\'])\n            with Image.open(directory/\'input.png\') as image:source=image.convert(\'RGB\').copy()\n            conditions=load_conditions(directory);masks=_cached_masks(z,case,source,conditions)\n            if masks is None and observations_archive:\n                with zipfile.ZipFile(observations_archive) as previous:masks=_cached_masks(previous,case,source,conditions)\n            if masks is None:\n                results.append(dict(case_id=case[\'case_id\'],status=\'deferred_no_saved_extraction\',diffusion_run=False))\n                continue\n            try:\n                given=None if scores is None else scores.get(case[\'case_id\'])\n                guide,rendered=build_target_plan(source,conditions,masks,scores=given,config=config)\n                save_plan(guide,rendered,directory/\'target_plan\')\n                results.append(dict(case_id=case[\'case_id\'],status=\'target_ready_cpu_only\',\n                    strokes=len(guide.report[\'strokes\']),selected_strokes=sum(s[\'edit_strength\']>0 for s in guide.report[\'strokes\']),\n                    score_source=\'provided\' if given is not None else \'geometry_proxy_v1\',diffusion_run=False))\n            except ValueError as exc:\n                results.append(dict(case_id=case[\'case_id\'],status=\'deferred_invalid_plan\',error=str(exc),diffusion_run=False))\n            json_write(root/\'target_plan_summary.json\',results)\n    json_write(root/\'target_plan_summary.json\',results)\n    return results\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--archive\',required=True);parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--observations-archive\');parser.add_argument(\'--case\',action=\'append\',dest=\'cases\')\n    parser.add_argument(\'--scores\',help=\'JSON: case_id -> four lists of consistency scores in [0,100], in stroke-ID order\')\n    args=parser.parse_args()\n    given=json.loads(Path(args.scores).read_text(encoding=\'utf-8\')) if args.scores else None\n    print(json.dumps(prepare_archive(args.archive,args.output,case_ids=args.cases,\n        observations_archive=args.observations_archive,scores=given),indent=2))\n'}
SOURCE_HASHES={'stroke_model/reference_model.py': 'df3445a74e2543ee9e86037b03316650918231f99da41dcf1efb0fb81b7f597f', 'stroke_model/stroke_structure.py': '9d6b782def7fa36195db5d45410fd689ec130a14fdf626f23b6da402c767519f', 'stroke_model/source_structure.py': '1687ff6109a7e7f2ecc9a89335d17fa8583dc7bb65227fb261ec2c4bf0722d9c', 'stroke_model/target_stroke_plan.py': 'b2a92f4665654e7b7852acffd558d27b1193b6b209eff7d9675a863e6fc2cca6', 'diffusion_model/stroke_projected_diffusion.py': 'e1bf4fc6e10a3066f2180d2c61753b55331534afca798d6708f8e815981cbd6a', 'repair_audit/stage1.py': '83a898997f8b75a68ffff25675a5903064465fe7ad3188e4a9a70d549488a94f', 'repair_audit/target_guided_repair.py': '43db18e11b9d74a1967af45a955a1f32ad931d2748455522aba663d32f37a4d3'}
RUNTIME = Path('/content/hsqm_target_guided_runtime')
for name, code in RUNTIME_HELPERS.items():
    if hashlib.sha256(code.encode()).hexdigest() != SOURCE_HASHES[name]:
        raise ValueError('Embedded source hash mismatch: '+name)
    target = RUNTIME/name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(code, encoding='utf-8')
    (target.parent/'__init__.py').touch()
if str(RUNTIME) not in sys.path: sys.path.insert(0, str(RUNTIME))
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from google.colab import drive, files
from repair_audit.stage1 import read_archive, load_conditions, file_hash, json_write
from repair_audit.target_guided_repair import repair_generated_sheet, _cached_masks, extract_sheet_masks
from stroke_model.target_stroke_plan import TargetPlanConfig
if not torch.cuda.is_available():
    raise RuntimeError('런타임 유형을 GPU로 바꾼 뒤 다시 실행하세요.')
print('GPU:', torch.cuda.get_device_name(0))


## 입력과 설정
- ARCHIVE를 비워 두면 ZIP 업로드 창이 열립니다. best.pt는 업로드하지 말고 Drive 경로를 확인하세요.
- CASE_IDS 기본값은 첫 시트이며 정상 대조군이 자동 포함됩니다. 전체 비교는 None으로 변경합니다.
- SCORES_JSON은 선택 사항: `{case_id: [[첫칸 획별 0~100점], [...], [...], [...]]}`.
- 기존 ZIP의 저장 추출을 우선 사용하고, 없는 정상 대조군 등은 같은 체크포인트로 추출합니다.
- Drive에 사례 완료/실패/수동 중단 시 ZIP을 백업합니다. 강제 연결 종료는 마지막 백업까지만 보존됩니다.


In [ ]:
# 2. 입력 선택 및 Drive 연결
ARCHIVE = ''
CHECKPOINT = '/content/drive/Shareddrives/2026 자율연구/HSQM/v2_topology_runs/20261104_214705_208224/best.pt'
CASE_IDS = ['group_000_page_001']  # None이면 전체 생성 시트 + 정상 대조군
SCORES_JSON = ''  # 선택: Drive의 획별 점수 JSON 경로
DRIVE_RESULT_DIR = '/content/drive/MyDrive/HSQM_target_guided_results'
REPAIR_STEPS, REPAIR_STRENGTH, REPAIR_SEED = 60, .25, 2026
PLAN_CONFIG = dict(keep_score=90., full_edit_score=40., maximum_step_blend=.8)
MODEL_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
REVISION = '451f4fe16113bff5a5d2269ed5ad43b0592e9a14'
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
if not ARCHIVE:
    print('4단계 결과 ZIP 하나를 선택하세요.')
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('ZIP 하나만 선택하세요.')
    ARCHIVE = str(Path(next(iter(uploaded))).resolve())
    del uploaded
if not Path(CHECKPOINT).is_file():
    raise FileNotFoundError('CHECKPOINT를 기존 best.pt 경로로 수정한 뒤 2번을 다시 실행하세요.')
OUTPUT = Path('/content')/('target_guided_repair_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
CASES, EXPERIMENT = read_archive(ARCHIVE, OUTPUT, CASE_IDS, include_controls=True)
if file_hash(CHECKPOINT) != EXPERIMENT['checkpoint_sha256'].lower():
    raise ValueError('기존 ZIP과 best.pt SHA256이 다릅니다.')
SCORES = json.loads(Path(SCORES_JSON).read_text(encoding='utf-8')) if SCORES_JSON else {}
json_write(OUTPUT/'runtime_manifest.json', dict(source_hashes=SOURCE_HASHES,
    model_id=MODEL_ID, revision=REVISION, checkpoint_sha256=file_hash(CHECKPOINT),
    input_archive_sha256=file_hash(ARCHIVE), torch_version=torch.__version__,
    gpu=torch.cuda.get_device_name(0), scores=SCORES,
    settings=dict(steps=REPAIR_STEPS, strength=REPAIR_STRENGTH, seed=REPAIR_SEED, plan=PLAN_CONFIG)))
for case in CASES:
    print(case['case_id'], case['kind'])
    display(Image.open(Path(case['directory'])/'input.png').resize((288,288)))
print('저장 위치:', OUTPUT)


In [ ]:
# 3. 기존 획 추출기와 SD1.5 로딩 (추가 학습 없음)
from stroke_model.reference_model import ReferenceStrokeModel
from diffusers import StableDiffusionImg2ImgPipeline, DDIMScheduler
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
if state.get('architecture') != 'reference_v2_topology_loss_v1':
    raise ValueError('reference_v2_topology_loss_v1 체크포인트가 필요합니다.')
extractor = ReferenceStrokeModel(image_size=state['image_size'], width=state['width'], pretrained=False)
extractor.load_state_dict(state['model'], strict=True)
for case in CASES:
    if any(c['reference'].shape[-2:] != (state['image_size'],state['image_size'])
           for c in load_conditions(case['directory'])):
        raise ValueError('조건 이미지와 추출기 해상도가 다릅니다.')
extractor.eval().requires_grad_(False)  # 추출할 때만 GPU에 올림
del state; gc.collect(); torch.cuda.empty_cache()
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(MODEL_ID, revision=REVISION,
    torch_dtype=torch.float16, use_safetensors=True)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config, clip_sample=False, thresholding=False)
pipe.to('cuda'); pipe.enable_attention_slicing(); pipe.enable_vae_slicing()
for module in (pipe.unet, pipe.vae, pipe.text_encoder): module.eval().requires_grad_(False)
pipe.vae.to(dtype=torch.float32)
json_write(OUTPUT/'scheduler_config.json',dict(pipe.scheduler.config))
import importlib.metadata
json_write(OUTPUT/'versions.json',{name:importlib.metadata.version(name) for name in
    ('torch','diffusers','transformers','timm','numpy','scipy','scikit-image')})
print('로딩 완료. 4번 실행.')


In [ ]:
# 4. 두 경로 비교. 같은 세션에서 재실행하면 완료 사례는 건너뜀
def save_snapshot():
    archive = shutil.make_archive(str(OUTPUT)+'_checkpoint', 'zip', OUTPUT)
    if DRIVE_RESULT_DIR:
        destination = Path(DRIVE_RESULT_DIR); destination.mkdir(parents=True, exist_ok=True)
        shutil.copy2(archive, destination/Path(archive).name)
    return archive

def show_comparison(case):
    root = Path(case['directory']); repair = root/'target_repair'
    panels = [('input', root/'input.png'), ('target: NOT diffusion', repair/'target_plan/aligned_target.png'),
              ('redenoise only', repair/'reverse/redenoise_only.png'),
              ('target guided', repair/'reverse/stroke_projected.png')]
    side = 288; image = Image.new('RGB',(side*4,side+30),'white'); draw = ImageDraw.Draw(image)
    for i,(label,path) in enumerate(panels):
        draw.text((i*side+5,8),label,fill='black')
        with Image.open(path) as panel: image.paste(panel.convert('RGB').resize((side,side)),(i*side,30))
    image.save(root/'diffusion_comparison.png'); display(image)

for case in CASES:
    root = Path(case['directory']); result = root/'target_repair'
    try:
        if (result/'pipeline.json').is_file():
            print('완료 사례 건너뜀:',case['case_id']); show_comparison(case); continue
        if result.exists():
            result.rename(root/('interrupted_attempt_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f')))
        source = Image.open(root/'input.png').convert('RGB')
        conditions = load_conditions(root)
        with zipfile.ZipFile(ARCHIVE) as z: masks = _cached_masks(z,case,source,conditions)
        if masks is None:
            print(case['case_id'],'저장 추출 없음: 기존 모델로 추출',flush=True)
            try:
                extractor.to('cuda')
                masks = extract_sheet_masks(source,conditions,extractor)
            finally:
                extractor.to('cpu'); gc.collect(); torch.cuda.empty_cache()
        else: print(case['case_id'],'동일 입력의 저장 추출 재사용',flush=True)
        cache = root/'cached_extraction'; cache.mkdir(exist_ok=True)
        import numpy as np
        for i,mask in enumerate(masks,1): np.savez_compressed(cache/f'slot_{i}.npz',extracted_masks=mask)
        arm_clock = {}
        def progress(arm,step,total,row):
            now = time.monotonic()
            arm_clock.setdefault(arm,now)
            if step == 1 or step%3 == 0 or step == total:
                elapsed = now-arm_clock[arm]
                eta = elapsed/(step-1)*(total-step) if step>1 else None
                print(case['case_id'],arm,f'{step}/{total}',
                      '남은 초≈'+str(round(eta)) if eta is not None else '시간 측정 중',flush=True)
        guide, report = repair_generated_sheet(pipe,extractor,source,conditions,result,
            cached_masks=masks,scores=SCORES.get(case['case_id']),config=TargetPlanConfig(**PLAN_CONFIG),
            steps=REPAIR_STEPS,strength=REPAIR_STRENGTH,seed=REPAIR_SEED,
            prompt=case.get('prompt',''),negative_prompt=case.get('negative_prompt',''),progress=progress)
        selected = sum(s['edit_strength']>0 for s in guide.report['strokes'])
        print('수정 대상:',selected,'/',len(guide.report['strokes']), '획; 실제 스텝:',report['actual_steps'])
        show_comparison(case)
        (root/'failure.json').unlink(missing_ok=True)
    except KeyboardInterrupt:
        print('수동 중단: 부분 결과 백업. 같은 세션에서 4번을 다시 실행하면 미완료 사례부터 재계산합니다.')
        break
    except Exception as exc:
        json_write(root/'failure.json',dict(error=str(exc),traceback=traceback.format_exc()))
        print('실패:',case['case_id'],str(exc),flush=True)
    finally:
        snapshot = save_snapshot()
        print('백업:',snapshot,flush=True)
        gc.collect(); torch.cuda.empty_cache()
print('현재 완료',sum((Path(c['directory'])/'target_repair/pipeline.json').is_file() for c in CASES),'/',len(CASES))


## 결과 확인
- diffusion_comparison.png에서 정상 대조군도 함께 확인하세요. 목표 이미지는 확산 결과가 아닙니다.
- 획이 얇아지는지, 정상 획이 변하는지, 오류 획이 목표 위치로 이동하는지, 배경이 자연스러운지 봅니다.
- target_plan/plan.json에 관측 두께·정합·점수·수정 강도가 있습니다.
- reverse/report.json에 동일 초기 latent 해시와 경로별 진단이 있습니다. 진단값 감소가 글자 정답률을 뜻하지는 않습니다.
- 이번 가이드는 매 스텝 VAE를 왕복합니다. T4에서도 실행 가능하도록 구성했지만 속도·메모리·품질은 실제 GPU 검증 전입니다.
- 재연결로 런타임이 사라지면 Drive 백업을 확보하세요. 이 노트북은 **세션 간 중간 latent 재개를 지원하지 않습니다**.


In [ ]:
# 5. 결과 ZIP 다운로드 (완료/부분 결과 모두 가능)
archive = save_snapshot()
print('Drive:',DRIVE_RESULT_DIR)
print('다운로드 후 이 ZIP을 보내주세요:',archive)
files.download(archive)
